# Causal Intervention Framework for VAE Mechanistic Interpretability
## Complete Experimental Pipeline

**Paper**: *Causal Intervention Framework for Variational Auto Encoder Mechanistic Interpretability*  
**Author**: Dip Roy, IIT Patna  

This notebook implements the complete experimental pipeline including:
- 5 real-world datasets (dSprites, 3DShapes, MPI3D, CelebA, SmallNORB)
- 6 VAE architectures (Standard, β-VAE, FactorVAE, β-TC-VAE, DIP-VAE-II, VQ-VAE)
- 4-level causal intervention framework
- Standard disentanglement metrics (DCI, MIG, SAP, β-metric, FactorVAE metric)
- Novel causal metrics (CES, Specificity, Modularity, Polysemanticity)
- Publication-grade visualizations saved to W&B and locally
- Full checkpointing and reproducibility

**Hardware**: Designed for RunPod with RTX 5090 / L40S GPU

## 1. Configuration & Mode Selection
Toggle `TEST_MODE = True` to run a quick validation pass with reduced data and epochs.  
Set `TEST_MODE = False` for the full experiment.

In [ ]:
# ============================================================
# EXPERIMENT MODE CONFIGURATION
# ============================================================
TEST_MODE = True  # Set to False for full experiment

# --- Experiment Identifiers ---
EXPERIMENT_NAME = "vae-mechanistic-interpretability"
WANDB_PROJECT = "vae-causal-interventions"
WANDB_ENTITY = None  # Set to your W&B username/org

# --- Output Directories ---
import os
BASE_DIR = os.path.expanduser("~/vae_mi_experiments")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
DATA_DIR = os.path.join(BASE_DIR, "data")

for d in [CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Mode-dependent hyperparameters ---
if TEST_MODE:
    NUM_SEEDS = 1
    TRAIN_EPOCHS = 3
    BATCH_SIZE = 64
    MAX_TRAIN_SAMPLES = 5000
    MAX_EVAL_SAMPLES = 1000
    NUM_INTERVENTION_SAMPLES = 200
    LATENT_DIMS = 10
    BETA_SWEEP = [1.0, 4.0]
    GAMMA_SWEEP = [10.0, 40.0]
    DATASETS_TO_RUN = ["dsprites", "shapes3d", "mpi3d", "celeba", "smallnorb"]
    MODELS_TO_RUN = [
        "standard_vae", "beta_vae", "factor_vae",
        "beta_tc_vae", "dip_vae", "vq_vae"
    ]
    print(">>> RUNNING IN TEST MODE (all 6 models, all 5 datasets) <<<")
    print(f"    Seeds: {NUM_SEEDS}, Epochs: {TRAIN_EPOCHS}, Max samples: {MAX_TRAIN_SAMPLES}")
    print(f"    Models: {len(MODELS_TO_RUN)}, Datasets: {len(DATASETS_TO_RUN)}")
else:
    NUM_SEEDS = 3
    TRAIN_EPOCHS = 50
    BATCH_SIZE = 64
    MAX_TRAIN_SAMPLES = None  # Use full dataset
    MAX_EVAL_SAMPLES = 10000
    NUM_INTERVENTION_SAMPLES = 2000
    LATENT_DIMS = 10
    BETA_SWEEP = [1.0, 2.0, 4.0, 8.0, 16.0]
    GAMMA_SWEEP = [10.0, 20.0, 40.0, 80.0]
    DATASETS_TO_RUN = ["dsprites", "shapes3d", "mpi3d", "celeba", "smallnorb"]
    MODELS_TO_RUN = [
        "standard_vae", "beta_vae", "factor_vae",
        "beta_tc_vae", "dip_vae", "vq_vae"
    ]
    print(">>> RUNNING IN FULL MODE <<<")
    print(f"    Seeds: {NUM_SEEDS}, Epochs: {TRAIN_EPOCHS}, Datasets: {len(DATASETS_TO_RUN)}, Models: {len(MODELS_TO_RUN)}")

# --- Fixed hyperparameters ---
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
IMG_SIZE = 64
IMG_CHANNELS = {"dsprites": 1, "shapes3d": 3, "mpi3d": 3, "celeba": 3, "smallnorb": 1}
INTERVENTION_RANGE = (-3.0, 3.0)
INTERVENTION_STEPS = 11

# Model-specific defaults
MODEL_CONFIGS = {
    "standard_vae": {"type": "standard", "beta": 1.0},
    "beta_vae": {"type": "beta", "beta": 4.0},
    "factor_vae": {"type": "factor", "gamma": 10.0},
    "beta_tc_vae": {"type": "beta_tc", "beta": 1.0, "alpha": 1.0, "gamma_tc": 1.0},
    "dip_vae": {"type": "dip", "lambda_od": 10.0, "lambda_d": 100.0},
    "vq_vae": {"type": "vq", "num_embeddings": 512, "commitment_cost": 0.25},
}

print(f"\nOutput directories:")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Results:     {RESULTS_DIR}")
print(f"  Figures:     {FIGURES_DIR}")
print(f"  Data:        {DATA_DIR}")

## 2. Environment Setup

In [ ]:
# ============================================================
# PACKAGE INSTALLATION
# ============================================================
import subprocess, sys, importlib

def run_pip(args_str):
    subprocess.check_call(
        [sys.executable, "-m", "pip"] + args_str.split() + ["-q", "--root-user-action=ignore"]
    )

# --- Step 1: Check if typing_extensions needs upgrade (common RunPod issue) ---
needs_restart = False
try:
    from typing_extensions import Sentinel  # Required by pydantic_core >= 2.33
    print("typing_extensions is up to date.")
except (ImportError, AttributeError):
    print("Upgrading typing_extensions (required by wandb/pydantic)...")
    run_pip("install --upgrade typing_extensions pydantic pydantic-core")
    needs_restart = True

if needs_restart:
    print("\n" + "="*60)
    print("KERNEL RESTART REQUIRED")
    print("typing_extensions was upgraded but the old version is still")
    print("cached in this kernel session.")
    print("")
    print(">>> Please restart the kernel now, then re-run ALL cells. <<<")
    print("="*60)
    # Auto-restart in Jupyter if possible
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        pass
    raise SystemExit("Restart kernel and re-run.")

# --- Step 2: Install PyTorch ---
print("Installing PyTorch (CUDA 12.4)...")
try:
    import torch
    print(f"  PyTorch {torch.__version__} already installed, CUDA: {torch.cuda.is_available()}")
except ImportError:
    try:
        run_pip("install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")
    except Exception:
        print("  CUDA 12.4 wheel not found, installing default...")
        run_pip("install torch torchvision torchaudio")

# --- Step 3: Install experiment dependencies ---
# Check numpy/scipy compatibility BEFORE installing other packages
import numpy as _np_check
_np_ver_before = _np_check.__version__

# Upgrade numpy and scipy together to avoid ABI mismatch
run_pip("install --upgrade numpy scipy")

print("Installing experiment dependencies...")
run_pip("install wandb matplotlib seaborn scikit-learn h5py pillow tqdm pandas gdown datasets huggingface_hub")

# Check if numpy was upgraded (old version still loaded in memory)
from importlib.metadata import version as pkg_version
_np_ver_disk = pkg_version('numpy')
if _np_ver_before != _np_ver_disk:
    print(f"\nnumpy upgraded: {_np_ver_before} -> {_np_ver_disk}")
    print("="*60)
    print("KERNEL RESTART REQUIRED")
    print("numpy was upgraded but the old version is still in memory.")
    print("")
    print(">>> Please restart the kernel now, then re-run ALL cells. <<<")
    print("="*60)
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        pass
    raise SystemExit("Restart kernel and re-run.")

# --- Step 4: Verify ---
print("Verifying imports...")
from importlib.metadata import version as pkg_version
import wandb, torch
from scipy import stats  # This is what was failing
print(f"  torch={torch.__version__} (CUDA={torch.cuda.is_available()})")
print(f"  wandb={wandb.__version__}")
print(f"  numpy={pkg_version('numpy')}, scipy={pkg_version('scipy')}")
print(f"  typing_extensions={pkg_version('typing-extensions')}")
print(f"  pydantic-core={pkg_version('pydantic-core')}")
print("\nAll packages ready!")

### W&B Authentication
Enter your Weights & Biases API key when prompted.  
Get your key from: https://wandb.ai/authorize

In [ ]:
import wandb
import os

# Prompt for W&B API key at runtime
wandb_key = input("Enter your W&B API key (from https://wandb.ai/authorize): ").strip()
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    wandb.login(key=wandb_key)
    print("W&B login successful!")
else:
    print("WARNING: No API key provided. W&B logging will run in offline mode.")
    os.environ["WANDB_MODE"] = "offline"
    wandb.login(anonymous="allow")

In [ ]:
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torchvision
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.special import entr
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mutual_info_score
from sklearn.cluster import KMeans

import wandb
import json
import time
import copy
import hashlib
import warnings
import h5py
from pathlib import Path
from collections import defaultdict
from itertools import combinations
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

# Set publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = sns.color_palette("Set2", 8)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Utility Functions

In [ ]:
def set_seed(seed):
    """Set all random seeds for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def save_checkpoint(model, optimizer, epoch, loss, path, extra=None):
    """Save model checkpoint."""
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }
    if extra:
        state.update(extra)
    torch.save(state, path)

def load_checkpoint(model, path, optimizer=None, device=DEVICE):
    """Load model checkpoint."""
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint.get('epoch', 0), checkpoint.get('loss', float('inf'))

def get_checkpoint_path(model_name, dataset_name, seed):
    return os.path.join(CHECKPOINT_DIR, f"{model_name}_{dataset_name}_seed{seed}.pt")

def save_figure(fig, name, subdirectory=None):
    """Save figure locally, display in notebook, and log to W&B."""
    if subdirectory:
        save_dir = os.path.join(FIGURES_DIR, subdirectory)
        os.makedirs(save_dir, exist_ok=True)
    else:
        save_dir = FIGURES_DIR
    
    path_png = os.path.join(save_dir, f"{name}.png")
    path_pdf = os.path.join(save_dir, f"{name}.pdf")
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    fig.savefig(path_pdf, bbox_inches='tight')
    
    if wandb.run is not None:
        wandb.log({f"figures/{name}": wandb.Image(path_png)})
    
    # Display in notebook before closing
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        pass
    
    plt.close(fig)
    print(f"    Saved: {path_png}")
    return path_png

def save_results(results_dict, name):
    """Save results dict as JSON locally and as W&B artifact."""
    path = os.path.join(RESULTS_DIR, f"{name}.json")
    
    # Convert numpy/torch types to native Python for JSON serialization
    def convert(obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        elif isinstance(obj, (np.floating,)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, dict):
            return {k: convert(v) for k, v in obj.items() 
                    if k != 'traversals'}  # Skip large traversal tensors
        elif isinstance(obj, (list, tuple)):
            # Skip lists of tensors (e.g., traversals)
            if len(obj) > 0 and isinstance(obj[0], torch.Tensor):
                return None
            return [convert(v) for v in obj]
        elif isinstance(obj, (bool,)):
            return obj
        elif isinstance(obj, (int, float, str, type(None))):
            return obj
        else:
            return str(obj)  # Fallback: convert unknown types to string
    
    with open(path, 'w') as f:
        json.dump(convert(results_dict), f, indent=2)
    
    if wandb.run is not None:
        wandb.save(path)
    return path

def log_metrics(metrics_dict, step=None, prefix=""):
    """Log metrics to W&B with optional prefix."""
    if wandb.run is not None:
        logged = {f"{prefix}/{k}" if prefix else k: v 
                  for k, v in metrics_dict.items()
                  if isinstance(v, (int, float, np.integer, np.floating))}
        if step is not None:
            wandb.log(logged, step=step)
        else:
            wandb.log(logged)

## 4. Dataset Loading

We use 5 real-world disentanglement benchmarks with increasing complexity.
Each dataset provides ground-truth factors of variation for evaluation.

In [ ]:
class DisentanglementDataset(Dataset):
    """Base class for disentanglement benchmark datasets."""
    
    def __init__(self, images, factors, factor_names, transform=None):
        self.images = images  # (N, C, H, W) float32 in [0, 1]
        self.factors = factors  # (N, K) factor values
        self.factor_names = factor_names
        self.transform = transform
        self.num_factors = len(factor_names)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]
        fac = self.factors[idx]
        if self.transform:
            img = self.transform(img)
        return img, fac


def load_dsprites(data_dir, max_samples=None):
    """Load dSprites dataset."""
    path = os.path.join(data_dir, "dsprites.npz")
    
    if not os.path.exists(path):
        print("Downloading dSprites...")
        import urllib.request
        
        # Multiple URLs - the repo moved from deepmind -> google-deepmind,
        # and the actual filename is dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz
        urls = [
            "https://raw.githubusercontent.com/google-deepmind/dsprites-dataset/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz",
            "https://github.com/google-deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz",
            "https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz",
        ]
        
        downloaded = False
        for url in urls:
            try:
                print(f"  Trying: {url[:80]}...")
                urllib.request.urlretrieve(url, path)
                # Verify the file is valid
                if os.path.getsize(path) > 1_000_000:  # Should be ~2.5MB+
                    downloaded = True
                    print(f"  Downloaded ({os.path.getsize(path) / 1e6:.1f} MB)")
                    break
                else:
                    os.remove(path)
                    print("  File too small, trying next URL...")
            except Exception as e:
                print(f"  Failed: {e}")
                if os.path.exists(path):
                    os.remove(path)
        
        if not downloaded:
            # Try gdown as last resort (GitHub LFS files sometimes need this)
            try:
                import gdown
                print("  Trying gdown...")
                gdown.download(
                    "https://github.com/google-deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz",
                    path, quiet=False
                )
                downloaded = os.path.exists(path) and os.path.getsize(path) > 1_000_000
            except Exception as e:
                print(f"  gdown failed: {e}")
        
        if not downloaded:
            raise RuntimeError(
                "Could not download dSprites. Please download manually:\n"
                "  1. Go to: https://github.com/google-deepmind/dsprites-dataset\n"
                "  2. Download dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz\n"
                f"  3. Save as: {path}\n"
                "  Then re-run this cell."
            )
    
    data = np.load(path, allow_pickle=True, encoding='latin1')
    images = data['imgs']  # (737280, 64, 64) binary
    factors = data['latents_values'][:, 1:]  # Remove first column (color, always 1)
    factor_names = ['shape', 'scale', 'orientation', 'posX', 'posY']
    
    # Subsample if needed
    if max_samples and max_samples < len(images):
        idx = np.random.choice(len(images), max_samples, replace=False)
        images = images[idx]
        factors = factors[idx]
    
    images = images.astype(np.float32)[:, np.newaxis, :, :]  # (N, 1, 64, 64)
    factors = factors.astype(np.float32)
    
    print(f"  dSprites loaded: {images.shape[0]} samples, {len(factor_names)} factors: {factor_names}")
    return DisentanglementDataset(images, factors, factor_names)


def load_shapes3d(data_dir, max_samples=None):
    """Load 3DShapes dataset."""
    path = os.path.join(data_dir, "3dshapes.h5")
    
    if not os.path.exists(path):
        print("Downloading 3DShapes...")
        url = "https://storage.googleapis.com/3d-shapes/3dshapes.h5"
        import urllib.request
        urllib.request.urlretrieve(url, path)
    
    with h5py.File(path, 'r') as f:
        images = f['images'][:]  # (480000, 64, 64, 3)
        factors = f['labels'][:]  # (480000, 6)
    
    factor_names = ['floor_hue', 'wall_hue', 'object_hue', 'scale', 'shape', 'orientation']
    
    if max_samples and max_samples < len(images):
        idx = np.random.choice(len(images), max_samples, replace=False)
        images = images[idx]
        factors = factors[idx]
    
    images = images.astype(np.float32).transpose(0, 3, 1, 2) / 255.0  # (N, 3, 64, 64)
    factors = factors.astype(np.float32)
    
    print(f"  3DShapes loaded: {images.shape[0]} samples, {len(factor_names)} factors: {factor_names}")
    return DisentanglementDataset(images, factors, factor_names)


def load_mpi3d(data_dir, max_samples=None):
    """Load MPI3D dataset via huggingface_hub (downloads raw npz, no loading script needed)."""
    path_real = os.path.join(data_dir, "mpi3d_real.npz")
    path_toy = os.path.join(data_dir, "mpi3d_toy.npz")
    
    factor_names = ['object_color', 'object_shape', 'object_size',
                    'camera_height', 'background_color', 'horizontal_axis', 'vertical_axis']
    n_values = [6, 6, 2, 3, 3, 40, 40]
    
    # Check local cache first
    path = None
    if os.path.exists(path_real):
        path = path_real
    elif os.path.exists(path_toy):
        path = path_toy
    
    if path is None:
        # Download raw npz from HuggingFace Hub (bypasses loading script)
        print("  Downloading MPI3D-toy from HuggingFace Hub...")
        from huggingface_hub import hf_hub_download
        
        try:
            path = hf_hub_download(
                repo_id="randall-lab/mpi3d-toy",
                filename="mpi3d_toy.npz",
                repo_type="dataset",
                local_dir=data_dir
            )
            print(f"  Downloaded to: {path}")
        except Exception as e1:
            print(f"  randall-lab failed: {e1}, trying cun-bjy/mpi3d_real...")
            try:
                path = hf_hub_download(
                    repo_id="cun-bjy/mpi3d_real",
                    filename="mpi3d_real.npz",
                    repo_type="dataset",
                    local_dir=data_dir
                )
            except Exception as e2:
                raise RuntimeError(
                    f"MPI3D download failed: {e2}\n"
                    "Please download manually:\n"
                    "  pip install huggingface_hub\n"
                    "  python -c \"from huggingface_hub import hf_hub_download; "
                    f"hf_hub_download('randall-lab/mpi3d-toy', 'mpi3d_toy.npz', repo_type='dataset', local_dir='{data_dir}')\""
                )
    
    print(f"  Loading from: {path}")
    data = np.load(path)
    images = data['images']
    print(f"  Raw images shape: {images.shape}, dtype: {images.dtype}")
    
    # Handle different storage formats from different HF repos
    # mpi3d_toy: (1036800, 64, 64, 3) — already correct
    # mpi3d_real from cun-bjy: may be flat or different shape
    if images.ndim == 1:
        # Flat array — reshape to (N, 64, 64, 3)
        total_pixels = 64 * 64 * 3
        n_images = len(images) // total_pixels
        images = images.reshape(n_images, 64, 64, 3)
        print(f"  Reshaped flat array to: {images.shape}")
    elif images.ndim == 2:
        # (N, flat_pixels)
        n_images = images.shape[0]
        pixels_per_img = images.shape[1]
        # Determine resolution
        for res in [64, 128, 96]:
            if pixels_per_img == res * res * 3:
                images = images.reshape(n_images, res, res, 3)
                print(f"  Reshaped 2D array to: {images.shape}")
                break
        else:
            # Try square root approach
            side = int(np.sqrt(pixels_per_img / 3))
            images = images.reshape(n_images, side, side, 3)
            print(f"  Reshaped 2D array to: {images.shape}")
    elif images.ndim == 4:
        # Detect channels-first (N, 3, H, W) vs channels-last (N, H, W, 3)
        if images.shape[1] in [1, 3] and images.shape[1] < images.shape[2]:
            print(f"  Channels-first detected: {images.shape} -> transposing to channels-last")
            images = images.transpose(0, 2, 3, 1)  # (N, H, W, C)
        print(f"  4D array, shape: {images.shape}")
    
    # Resize to IMG_SIZE x IMG_SIZE if not already correct
    if images.shape[1] != IMG_SIZE or images.shape[2] != IMG_SIZE:
        from PIL import Image as PILImage
        print(f"  Resizing from {images.shape[1]}x{images.shape[2]} to {IMG_SIZE}x{IMG_SIZE}...")
        n = min(len(images), max_samples or len(images))
        resized = np.zeros((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        for i in range(n):
            pil_img = PILImage.fromarray(images[i].astype(np.uint8) if images.dtype != np.uint8 else images[i])
            resized[i] = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE), PILImage.BILINEAR))
        images = resized
        print(f"  Resized to: {images.shape}")
    
    total = min(int(np.prod(n_values)), len(images))
    factors = np.zeros((total, len(factor_names)), dtype=np.float32)
    idx = np.arange(total)
    for i in range(len(factor_names) - 1, -1, -1):
        factors[:, i] = idx % n_values[i]
        idx = idx // n_values[i]
    
    images = images[:total]
    if max_samples and max_samples < len(images):
        sel = np.random.choice(len(images), max_samples, replace=False)
        images, factors = images[sel], factors[sel]
    
    images = images.astype(np.float32).transpose(0, 3, 1, 2) / 255.0
    print(f"  MPI3D loaded: {images.shape[0]} samples, {len(factor_names)} factors: {factor_names}")
    return DisentanglementDataset(images, factors, factor_names)





def load_celeba(data_dir, max_samples=None):
    """Load CelebA dataset via HuggingFace (no Google Drive needed)."""
    
    selected_attrs = ['Male', 'Smiling', 'Young', 'Eyeglasses', 'Bald',
                      'Bangs', 'Black_Hair', 'Blond_Hair', 'Heavy_Makeup', 'Pale_Skin']
    
    # Try torchvision first (uses local cache if already downloaded)
    try:
        transform = transforms.Compose([
            transforms.CenterCrop(140),
            transforms.Resize(IMG_SIZE),
            transforms.ToTensor(),
        ])
        tv_dataset = torchvision.datasets.CelebA(
            root=data_dir, split='train', download=False, transform=transform
        )
        print("  CelebA loaded from local torchvision cache")
        
        attr_names = tv_dataset.attr_names if hasattr(tv_dataset, 'attr_names') else None
        if attr_names:
            attr_indices = [attr_names.index(a) for a in selected_attrs if a in attr_names]
        else:
            attr_indices = list(range(10))
        
        n = min(max_samples or len(tv_dataset), len(tv_dataset))
        images_list, factors_list = [], []
        loader = DataLoader(tv_dataset, batch_size=256, shuffle=False, num_workers=4)
        count = 0
        for batch_imgs, batch_attrs in tqdm(loader, desc="  Loading CelebA"):
            images_list.append(batch_imgs.numpy())
            factors_list.append(batch_attrs[:, attr_indices].numpy().astype(np.float32))
            count += batch_imgs.shape[0]
            if count >= n:
                break
        
        images = np.concatenate(images_list, axis=0)[:n]
        factors = np.concatenate(factors_list, axis=0)[:n]
        
        print(f"  CelebA loaded: {images.shape[0]} samples, {len(selected_attrs)} factors")
        return DisentanglementDataset(images, factors, selected_attrs)
    except Exception:
        pass  # Fall through to HuggingFace
    
    # Primary: Download from HuggingFace (no Google Drive rate limits)
    print("  Downloading CelebA from HuggingFace...")
    from datasets import load_dataset as hf_load
    
    # Try multiple HF repos (some have Parquet, some have loading scripts)
    hf_ds = None
    hf_repos = ["flwrlabs/celeba", "nielsr/CelebA-faces", "student/celebA"]
    for repo in hf_repos:
        try:
            print(f"    Trying {repo}...")
            hf_ds = hf_load(repo, split="train")
            print(f"    Loaded from {repo}")
            break
        except Exception as e:
            print(f"    {repo} failed: {e}")
    
    if hf_ds is None:
        raise RuntimeError(
            "CelebA auto-download failed from all HuggingFace sources.\n"
            "Please download manually via:\n"
            "  pip install datasets\n"
            "  python -c \"from datasets import load_dataset; "
            "ds = load_dataset('flwrlabs/celeba', split='train'); print(len(ds))\""
        )
    
    n = min(max_samples or len(hf_ds), len(hf_ds))
    if max_samples and max_samples < len(hf_ds):
        hf_ds = hf_ds.shuffle(seed=42).select(range(n))
    
    images_list = []
    factors_list = []
    
    for i in tqdm(range(len(hf_ds)), desc="  Processing CelebA"):
        sample = hf_ds[i]
        img = sample['image']
        # Center crop + resize to 64x64
        w, h = img.size
        crop = min(w, h)
        left = (w - crop) // 2
        top = (h - crop) // 2
        img = img.crop((left, top, left + crop, top + crop)).resize((64, 64))
        img_np = np.array(img, dtype=np.float32) / 255.0
        if img_np.ndim == 2:
            img_np = np.stack([img_np]*3, axis=-1)
        images_list.append(img_np.transpose(2, 0, 1))  # (3, 64, 64)
        
        # Extract attributes
        attr_vals = []
        for attr in selected_attrs:
            val = sample.get(attr, False)
            attr_vals.append(float(val) if isinstance(val, bool) else float(val))
        factors_list.append(attr_vals)
    
    images = np.array(images_list, dtype=np.float32)
    factors = np.array(factors_list, dtype=np.float32)
    
    print(f"  CelebA loaded: {images.shape[0]} samples, {len(selected_attrs)} factors")
    return DisentanglementDataset(images, factors, selected_attrs)




def load_smallnorb(data_dir, max_samples=None):
    """Load SmallNORB dataset (auto-downloads from NYU)."""
    import struct
    import gzip
    
    norb_dir = os.path.join(data_dir, "smallnorb")
    os.makedirs(norb_dir, exist_ok=True)
    
    base_url = "https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/"
    files = {
        "train_dat": "smallnorb-5x46789x9x18x6x2x96x96-training-dat.mat.gz",
        "train_cat": "smallnorb-5x46789x9x18x6x2x96x96-training-cat.mat.gz",
        "train_info": "smallnorb-5x46789x9x18x6x2x96x96-training-info.mat.gz",
        "test_dat": "smallnorb-5x01235x9x18x6x2x96x96-testing-dat.mat.gz",
        "test_cat": "smallnorb-5x01235x9x18x6x2x96x96-testing-cat.mat.gz",
        "test_info": "smallnorb-5x01235x9x18x6x2x96x96-testing-info.mat.gz",
    }
    
    def _download_norb_file(fname, url_base, out_dir):
        fpath = os.path.join(out_dir, fname)
        if not os.path.exists(fpath):
            print(f"    Downloading {fname}...")
            import urllib.request
            urllib.request.urlretrieve(url_base + fname, fpath)
        return fpath
    
    def _read_norb_mat(fpath):
        """Read a SmallNORB .mat.gz file (custom binary format)."""
        with gzip.open(fpath, 'rb') as f:
            # Read header: magic number (4 bytes)
            magic = struct.unpack('<I', f.read(4))[0]
            # Number of dimensions (4 bytes)
            ndim = struct.unpack('<I', f.read(4))[0]
            # Dimension sizes (ndim * 4 bytes, padded to multiple of 4)
            n_read = max(ndim, 3)  # Always reads at least 3 dim values
            dims = [struct.unpack('<I', f.read(4))[0] for _ in range(n_read)]
            dims = dims[:ndim]
            
            total_elements = 1
            for d in dims:
                total_elements *= d
            
            # Determine data type from magic number
            if magic == 0x1E3D4C55:  # uint8 (images)
                data = np.frombuffer(f.read(total_elements), dtype=np.uint8)
            elif magic == 0x1E3D4C54:  # int32 (labels/info)
                data = np.frombuffer(f.read(total_elements * 4), dtype=np.int32)
            else:
                raise ValueError(f"Unknown magic number: {magic:#010x}")
            
            return data.reshape(dims)
    
    try:
        # Download all files
        for key, fname in files.items():
            _download_norb_file(fname, base_url, norb_dir)
        
        # Parse training data
        print("  Parsing SmallNORB training data...")
        train_dat_path = os.path.join(norb_dir, files["train_dat"])
        train_cat_path = os.path.join(norb_dir, files["train_cat"])
        train_info_path = os.path.join(norb_dir, files["train_info"])
        
        images_raw = _read_norb_mat(train_dat_path)  # (24300, 2, 96, 96)
        categories = _read_norb_mat(train_cat_path)   # (24300,)
        info = _read_norb_mat(train_info_path)         # (24300, 4): instance, elevation, azimuth, lighting
        
        # Use only first stereo image
        images = images_raw[:, 0, :, :]  # (24300, 96, 96)
        
        # Resize to 64x64
        from PIL import Image as PILImage
        images_resized = np.zeros((len(images), 64, 64), dtype=np.float32)
        for i in range(len(images)):
            pil_img = PILImage.fromarray(images[i])
            pil_img = pil_img.resize((64, 64), PILImage.BILINEAR)
            images_resized[i] = np.array(pil_img, dtype=np.float32) / 255.0
        
        images = images_resized[:, np.newaxis, :, :]  # (N, 1, 64, 64)
        
        # Build factors: category, instance, elevation, azimuth, lighting
        factor_names = ['category', 'instance', 'elevation', 'azimuth', 'lighting']
        factors = np.zeros((len(categories), 5), dtype=np.float32)
        factors[:, 0] = categories.astype(np.float32)
        factors[:, 1] = info[:, 0].astype(np.float32)  # instance
        factors[:, 2] = info[:, 1].astype(np.float32)  # elevation
        factors[:, 3] = info[:, 2].astype(np.float32)  # azimuth
        factors[:, 4] = info[:, 3].astype(np.float32)  # lighting
        
        if max_samples and max_samples < len(images):
            sel = np.random.choice(len(images), max_samples, replace=False)
            images = images[sel]
            factors = factors[sel]
        
        print(f"  SmallNORB loaded: {images.shape[0]} samples, {len(factor_names)} factors: {factor_names}")
        return DisentanglementDataset(images, factors, factor_names)
    
    except Exception as e:
        raise RuntimeError(
            f"SmallNORB download/parse failed: {e}\n"
            "Please download manually:\n"
            "  1. Go to: https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/\n"
            "  2. Download the training -dat.mat.gz, -cat.mat.gz, -info.mat.gz files\n"
            f"  3. Place them in: {norb_dir}\n"
            "  Then re-run this cell."
        )


def load_dataset(name, data_dir=DATA_DIR, max_samples=None):
    """Load a dataset by name. Downloads automatically; raises error if download fails."""
    loaders = {
        'dsprites': load_dsprites,
        'shapes3d': load_shapes3d,
        'mpi3d': load_mpi3d,
        'celeba': load_celeba,
        'smallnorb': load_smallnorb,
    }
    
    if name not in loaders:
        raise ValueError(f"Unknown dataset: {name}. Available: {list(loaders.keys())}")
    
    print(f"\nLoading {name}...")
    ds = loaders[name](data_dir, max_samples=max_samples)
    
    if ds is None:
        raise RuntimeError(f"Failed to load dataset {name} - this should not happen, check loader.")
    
    return ds


# Quick test
if TEST_MODE:
    _test_ds = load_dataset("dsprites", max_samples=100)
    print(f"  Quick test: images shape={_test_ds.images.shape}, factors shape={_test_ds.factors.shape}")
    print(f"  Factors: {_test_ds.factor_names}")
    print(f"  Image range: [{_test_ds.images.min():.3f}, {_test_ds.images.max():.3f}]")
    del _test_ds

## 5. VAE Model Definitions

We implement 6 VAE architectures with shared encoder/decoder backbones:
1. **Standard VAE** - baseline ELBO
2. **β-VAE** - higher KL penalty
3. **FactorVAE** - adversarial total correlation penalty
4. **β-TC-VAE** - decomposed KL (MI + TC + dim-wise KL)
5. **DIP-VAE-II** - covariance penalty on q(z)
6. **VQ-VAE** - discrete latent codes via vector quantization

In [ ]:
# ============================================================
# ENCODER / DECODER BACKBONES
# ============================================================

class ConvEncoder(nn.Module):
    """Shared convolutional encoder for all VAE variants."""
    
    def __init__(self, in_channels, latent_dim, hidden_dims=[32, 64, 128]):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dims = hidden_dims
        
        layers = []
        ch = in_channels
        for h_dim in hidden_dims:
            layers.append(nn.Sequential(
                nn.Conv2d(ch, h_dim, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(h_dim),
                nn.ReLU(inplace=True)
            ))
            ch = h_dim
        self.encoder = nn.Sequential(*layers)
        
        # Compute flat size
        dummy = torch.zeros(1, in_channels, IMG_SIZE, IMG_SIZE)
        with torch.no_grad():
            dummy_out = self.encoder(dummy)
        self.flat_size = int(np.prod(dummy_out.shape[1:]))
        self.spatial_shape = dummy_out.shape[1:]  # (C, H, W)
        
        self.fc_mu = nn.Linear(self.flat_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_size, latent_dim)
    
    def forward(self, x):
        h = self.encoder(x)
        h_flat = h.reshape(h.size(0), -1)
        mu = self.fc_mu(h_flat)
        logvar = self.fc_logvar(h_flat)
        return mu, logvar, h
    
    def get_layer_activations(self, x):
        """Return activations at each conv layer."""
        activations = {}
        h = x
        for i, layer in enumerate(self.encoder):
            h = layer(h)
            activations[f'encoder_conv_{i}'] = h.clone()
        activations['mu'] = self.fc_mu(h.reshape(h.size(0), -1))
        activations['logvar'] = self.fc_logvar(h.reshape(h.size(0), -1))
        return activations


class ConvDecoder(nn.Module):
    """Shared convolutional decoder for all VAE variants."""
    
    def __init__(self, latent_dim, out_channels, spatial_shape, hidden_dims=[128, 64, 32]):
        super().__init__()
        self.spatial_shape = spatial_shape
        flat_size = int(np.prod(spatial_shape))
        
        self.fc = nn.Linear(latent_dim, flat_size)
        
        layers = []
        for i in range(len(hidden_dims) - 1):
            layers.append(nn.Sequential(
                nn.ConvTranspose2d(hidden_dims[i], hidden_dims[i+1], kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(hidden_dims[i+1]),
                nn.ReLU(inplace=True)
            ))
        # Final layer
        layers.append(nn.Sequential(
            nn.ConvTranspose2d(hidden_dims[-1], out_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        ))
        self.decoder = nn.Sequential(*layers)
    
    def forward(self, z):
        h = self.fc(z)
        h = h.reshape(-1, *self.spatial_shape)
        return self.decoder(h)
    
    def get_layer_activations(self, z):
        """Return activations at each deconv layer."""
        activations = {}
        h = self.fc(z)
        h = h.reshape(-1, *self.spatial_shape)
        for i, layer in enumerate(self.decoder):
            h = layer(h)
            activations[f'decoder_conv_{i}'] = h.clone()
        return activations


# ============================================================
# VAE VARIANTS
# ============================================================

class BaseVAE(nn.Module):
    """Base VAE class with shared interface."""
    
    def __init__(self, in_channels, latent_dim, hidden_dims=[32, 64, 128]):
        super().__init__()
        self.latent_dim = latent_dim
        self.in_channels = in_channels
        self.encoder = ConvEncoder(in_channels, latent_dim, hidden_dims)
        self.decoder = ConvDecoder(latent_dim, in_channels, self.encoder.spatial_shape,
                                   hidden_dims=list(reversed(hidden_dims)))
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        mu, logvar, _ = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar, z
    
    def encode(self, x):
        mu, logvar, _ = self.encoder(x)
        return mu, logvar
    
    def decode(self, z):
        return self.decoder(z)
    
    def get_all_activations(self, x):
        """Get activations from all layers for intervention analysis."""
        enc_acts = self.encoder.get_layer_activations(x)
        mu, logvar = enc_acts['mu'], enc_acts['logvar']
        z = self.reparameterize(mu, logvar)
        dec_acts = self.decoder.get_layer_activations(z)
        enc_acts.update(dec_acts)
        enc_acts['z'] = z
        return enc_acts


class StandardVAE(BaseVAE):
    """Standard VAE with unit-weight KL."""
    
    def loss_function(self, x, recon, mu, logvar, **kwargs):
        recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        return {'loss': recon_loss + kl_loss, 'recon_loss': recon_loss, 'kl_loss': kl_loss}


class BetaVAE(BaseVAE):
    """β-VAE with weighted KL penalty."""
    
    def __init__(self, in_channels, latent_dim, beta=4.0, **kwargs):
        super().__init__(in_channels, latent_dim)
        self.beta = beta
    
    def loss_function(self, x, recon, mu, logvar, **kwargs):
        recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        return {'loss': recon_loss + self.beta * kl_loss, 'recon_loss': recon_loss, 'kl_loss': kl_loss}


class FactorVAEDiscriminator(nn.Module):
    """Discriminator for FactorVAE total correlation penalty."""
    
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1000), nn.LeakyReLU(0.2),
            nn.Linear(1000, 1000), nn.LeakyReLU(0.2),
            nn.Linear(1000, 1000), nn.LeakyReLU(0.2),
            nn.Linear(1000, 1000), nn.LeakyReLU(0.2),
            nn.Linear(1000, 1000), nn.LeakyReLU(0.2),
            nn.Linear(1000, 2)
        )
    
    def forward(self, z):
        return self.net(z)


class FactorVAE(BaseVAE):
    """FactorVAE with adversarial total correlation penalty."""
    
    def __init__(self, in_channels, latent_dim, gamma=40.0, **kwargs):
        super().__init__(in_channels, latent_dim)
        self.gamma = gamma
        self.discriminator = FactorVAEDiscriminator(latent_dim)
    
    def loss_function(self, x, recon, mu, logvar, z=None, d_z=None, **kwargs):
        recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        
        # TC penalty from discriminator (clamped to prevent explosion)
        tc_loss = torch.tensor(0.0, device=x.device)
        if d_z is not None:
            tc_loss = (d_z[:, 0] - d_z[:, 1]).mean()
            tc_loss = torch.clamp(tc_loss, -10.0, 10.0)  # Prevent divergence
        
        return {
            'loss': recon_loss + kl_loss + self.gamma * tc_loss,
            'recon_loss': recon_loss, 'kl_loss': kl_loss, 'tc_loss': tc_loss
        }
    
    def permute_latent(self, z):
        """Permute each dimension of z independently."""
        B, D = z.size()
        z_perm = z.clone()
        for d in range(D):
            perm = torch.randperm(B, device=z.device)
            z_perm[:, d] = z[perm, d]
        return z_perm


class BetaTCVAE(BaseVAE):
    """β-TC-VAE: decomposes KL into MI + TC + dim-wise KL."""
    
    def __init__(self, in_channels, latent_dim, alpha=1.0, beta=1.0, gamma_tc=1.0, **kwargs):
        super().__init__(in_channels, latent_dim)
        self.alpha = alpha  # MI weight
        self.beta_tc = beta  # TC weight  
        self.gamma_tc = gamma_tc  # dim-wise KL weight
    
    def _log_importance_weight(self, batch_size, dataset_size):
        """Compute log importance weight for minibatch stratified sampling."""
        N = dataset_size
        M = batch_size - 1
        strat_weight = (N - M) / (N * M)
        w = torch.Tensor(batch_size, batch_size).fill_(1 / M)
        w.reshape(-1)[::M+1] = 1 / N
        w.reshape(-1)[1::M+1] = strat_weight
        w[M-1, 0] = strat_weight
        return w.log()
    
    def loss_function(self, x, recon, mu, logvar, z=None, dataset_size=1, **kwargs):
        recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
        
        batch_size = x.size(0)
        
        # log q(z|x)
        log_qz_condx = -0.5 * (logvar + (z - mu).pow(2) / logvar.exp()).sum(dim=1)
        
        # log p(z)
        log_pz = -0.5 * z.pow(2).sum(dim=1)
        
        # log q(z) - approximate with minibatch
        _logqz = -0.5 * (
            logvar.unsqueeze(0) +
            (z.unsqueeze(1) - mu.unsqueeze(0)).pow(2) / logvar.unsqueeze(0).exp()
        )  # (B, B, D)
        
        logqz_prodmarginals = (torch.logsumexp(_logqz, dim=1) - np.log(batch_size)).sum(dim=1)
        logqz = torch.logsumexp(_logqz.sum(dim=2), dim=1) - np.log(batch_size)
        
        # Decomposed KL
        mi = (log_qz_condx - logqz).mean()
        tc = (logqz - logqz_prodmarginals).mean()
        dimwise_kl = (logqz_prodmarginals - log_pz).mean()
        
        loss = recon_loss + self.alpha * mi + self.beta_tc * tc + self.gamma_tc * dimwise_kl
        
        return {
            'loss': loss, 'recon_loss': recon_loss,
            'mi': mi, 'tc': tc, 'dimwise_kl': dimwise_kl,
            'kl_loss': mi + tc + dimwise_kl
        }


class DIPVAE(BaseVAE):
    """DIP-VAE-II: penalizes covariance of q(z)."""
    
    def __init__(self, in_channels, latent_dim, lambda_od=10.0, lambda_d=100.0, **kwargs):
        super().__init__(in_channels, latent_dim)
        self.lambda_od = lambda_od
        self.lambda_d = lambda_d
    
    def loss_function(self, x, recon, mu, logvar, **kwargs):
        recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        
        # DIP penalty on covariance of q(z)
        # Cov(mu) + E[diag(sigma^2)]
        mu_mean = mu.mean(dim=0)
        mu_centered = mu - mu_mean
        cov_mu = (mu_centered.t() @ mu_centered) / (mu.size(0) - 1)
        
        # Add expected variance
        cov_z = cov_mu + torch.diag(logvar.exp().mean(dim=0))
        
        # DIP-II: penalize both off-diagonal and diagonal
        cov_diag = torch.diag(cov_z)
        cov_offdiag = cov_z - torch.diag(cov_diag)
        
        dip_loss = (self.lambda_od * cov_offdiag.pow(2).sum() +
                    self.lambda_d * (cov_diag - 1).pow(2).sum())
        
        return {
            'loss': recon_loss + kl_loss + dip_loss,
            'recon_loss': recon_loss, 'kl_loss': kl_loss, 'dip_loss': dip_loss
        }


class VectorQuantizer(nn.Module):
    """Vector quantization layer for VQ-VAE."""
    
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)
    
    def forward(self, z_e):
        # z_e: (B, D, H, W) -> (B*H*W, D)
        B, D, H, W = z_e.shape
        z_flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)
        
        # Compute distances
        dists = (z_flat.pow(2).sum(dim=1, keepdim=True)
                 + self.embedding.weight.pow(2).sum(dim=1)
                 - 2 * z_flat @ self.embedding.weight.t())
        
        # Nearest embedding
        encoding_indices = dists.argmin(dim=1)
        z_q = self.embedding(encoding_indices).reshape(B, H, W, D).permute(0, 3, 1, 2)
        
        # Losses
        commitment_loss = F.mse_loss(z_e.detach(), z_q) 
        embedding_loss = F.mse_loss(z_e, z_q.detach())
        vq_loss = embedding_loss + self.commitment_cost * commitment_loss
        
        # Straight-through estimator
        z_q_st = z_e + (z_q - z_e).detach()
        
        return z_q_st, vq_loss, encoding_indices.reshape(B, H, W)


class VQVAE(nn.Module):
    """VQ-VAE with discrete latent codes."""
    
    def __init__(self, in_channels, latent_dim, num_embeddings=512,
                 commitment_cost=0.25, hidden_dims=[32, 64, 128], **kwargs):
        super().__init__()
        self.latent_dim = latent_dim
        self.in_channels = in_channels
        
        # Encoder (no mu/logvar - outputs continuous features)
        enc_layers = []
        ch = in_channels
        for h_dim in hidden_dims:
            enc_layers.append(nn.Sequential(
                nn.Conv2d(ch, h_dim, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(h_dim),
                nn.ReLU(inplace=True)
            ))
            ch = h_dim
        self.encoder_conv = nn.Sequential(*enc_layers)
        
        # Pre-VQ projection to latent_dim channels
        self.pre_vq = nn.Conv2d(hidden_dims[-1], latent_dim, kernel_size=1)
        
        # Vector quantizer
        self.vq = VectorQuantizer(num_embeddings, latent_dim, commitment_cost)
        
        # Post-VQ projection back
        self.post_vq = nn.Conv2d(latent_dim, hidden_dims[-1], kernel_size=1)
        
        # Decoder
        dec_layers = []
        rev_dims = list(reversed(hidden_dims))
        for i in range(len(rev_dims) - 1):
            dec_layers.append(nn.Sequential(
                nn.ConvTranspose2d(rev_dims[i], rev_dims[i+1], kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(rev_dims[i+1]),
                nn.ReLU(inplace=True)
            ))
        dec_layers.append(nn.Sequential(
            nn.ConvTranspose2d(rev_dims[-1], in_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        ))
        self.decoder_conv = nn.Sequential(*dec_layers)
    
    def forward(self, x):
        z_e = self.pre_vq(self.encoder_conv(x))
        z_q, vq_loss, indices = self.vq(z_e)
        recon = self.decoder_conv(self.post_vq(z_q))
        
        # For compatibility: create pseudo mu/logvar from spatial average
        mu = z_e.mean(dim=[2, 3])  # (B, latent_dim)
        logvar = torch.zeros_like(mu)
        
        return recon, mu, logvar, mu  # z = mu for VQ-VAE
    
    def encode(self, x):
        z_e = self.pre_vq(self.encoder_conv(x))
        mu = z_e.mean(dim=[2, 3])
        return mu, torch.zeros_like(mu)
    
    def decode(self, z):
        # Broadcast z to spatial dims
        B, D = z.shape
        dummy = torch.zeros(B, self.in_channels, IMG_SIZE, IMG_SIZE, device=z.device)
        z_e = self.pre_vq(self.encoder_conv(dummy * 0 + 0.5))
        H, W = z_e.shape[2], z_e.shape[3]
        z_spatial = z.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H, W)
        z_q, _, _ = self.vq(z_spatial)
        return self.decoder_conv(self.post_vq(z_q))
    
    def loss_function(self, x, recon, mu, logvar, **kwargs):
        recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
        # Recompute VQ loss
        z_e = self.pre_vq(self.encoder_conv(x))
        _, vq_loss, _ = self.vq(z_e)
        return {'loss': recon_loss + vq_loss, 'recon_loss': recon_loss, 'vq_loss': vq_loss, 'kl_loss': vq_loss}
    
    def get_all_activations(self, x):
        activations = {}
        h = x
        for i, layer in enumerate(self.encoder_conv):
            h = layer(h)
            activations[f'encoder_conv_{i}'] = h.clone()
        z_e = self.pre_vq(h)
        activations['mu'] = z_e.mean(dim=[2, 3])
        activations['logvar'] = torch.zeros_like(activations['mu'])
        activations['z'] = activations['mu']
        z_q, _, _ = self.vq(z_e)
        h = self.post_vq(z_q)
        for i, layer in enumerate(self.decoder_conv):
            h = layer(h)
            activations[f'decoder_conv_{i}'] = h.clone()
        return activations


def create_model(model_name, in_channels, latent_dim=LATENT_DIMS):
    """Factory function to create VAE model."""
    config = MODEL_CONFIGS[model_name]
    model_type = config['type']
    
    if model_type == 'standard':
        return StandardVAE(in_channels, latent_dim)
    elif model_type == 'beta':
        return BetaVAE(in_channels, latent_dim, beta=config['beta'])
    elif model_type == 'factor':
        return FactorVAE(in_channels, latent_dim, gamma=config['gamma'])
    elif model_type == 'beta_tc':
        return BetaTCVAE(in_channels, latent_dim, alpha=config['alpha'],
                         beta=config['beta'], gamma_tc=config['gamma_tc'])
    elif model_type == 'dip':
        return DIPVAE(in_channels, latent_dim,
                      lambda_od=config['lambda_od'], lambda_d=config['lambda_d'])
    elif model_type == 'vq':
        return VQVAE(in_channels, latent_dim,
                     num_embeddings=config['num_embeddings'],
                     commitment_cost=config['commitment_cost'])
    else:
        raise ValueError(f"Unknown model type: {model_type}")

# Quick test
print("Model parameter counts:")
for name in MODELS_TO_RUN:
    m = create_model(name, 1)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"  {name}: {n_params:,} params")
    del m

## 6. Training Infrastructure

Training loop with W&B logging, checkpointing, and learning rate scheduling.

In [ ]:
def train_vae(model, train_loader, val_loader, model_name, dataset_name, seed,
              epochs=TRAIN_EPOCHS, lr=LEARNING_RATE, device=DEVICE):
    """Train a VAE model with full logging and checkpointing."""
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # FactorVAE needs a discriminator optimizer
    disc_optimizer = None
    if isinstance(model, FactorVAE):
        disc_optimizer = optim.Adam(model.discriminator.parameters(), lr=5e-5)
    
    ckpt_path = get_checkpoint_path(model_name, dataset_name, seed)
    
    # Check for existing checkpoint
    start_epoch = 0
    history = {'train_loss': [], 'val_loss': [], 'kl_per_dim': []}
    if os.path.exists(ckpt_path):
        print(f"  Resuming from checkpoint: {ckpt_path}")
        start_epoch, _ = load_checkpoint(model, ckpt_path, optimizer, device)
        # Restore saved history from checkpoint
        ckpt_data = torch.load(ckpt_path, map_location=device, weights_only=False)
        if 'history' in ckpt_data and isinstance(ckpt_data['history'], dict):
            history = ckpt_data['history']
        start_epoch += 1
    
    dataset_size = len(train_loader.dataset)
    best_val_loss = float('inf')
    
    for epoch in range(start_epoch, epochs):
        model.train()
        epoch_losses = defaultdict(float)
        n_batches = 0
        
        for batch_idx, (data, factors) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            recon, mu, logvar, z = model(data)
            
            # Compute loss based on model type
            loss_kwargs = {'z': z, 'dataset_size': dataset_size}
            
            if isinstance(model, FactorVAE):
                # Warmup: only train discriminator after epoch 2
                # to let the VAE learn basic reconstructions first
                if epoch >= 2:
                    # Train discriminator
                    z_real = z.detach()
                    z_perm = model.permute_latent(z_real)
                    d_real = model.discriminator(z_real)
                    d_perm = model.discriminator(z_perm)
                    
                    d_loss = (F.cross_entropy(d_real, torch.zeros(z_real.size(0), dtype=torch.long, device=device)) +
                              F.cross_entropy(d_perm, torch.ones(z_perm.size(0), dtype=torch.long, device=device)))
                    
                    disc_optimizer.zero_grad()
                    d_loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.discriminator.parameters(), max_norm=5.0)
                    disc_optimizer.step()
                    
                    # VAE loss with TC from discriminator
                    d_z = model.discriminator(z)
                    loss_kwargs['d_z'] = d_z
            
            losses = model.loss_function(data, recon, mu, logvar, **loss_kwargs)
            
            # NaN detection - skip this batch
            if torch.isnan(losses['loss']) or torch.isinf(losses['loss']):
                continue
            
            losses['loss'].backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            for k, v in losses.items():
                epoch_losses[k] += v.item()
            n_batches += 1
        
        scheduler.step()
        
        # Average losses
        if n_batches == 0:
            print(f"  WARNING: All batches produced NaN at epoch {epoch}. Stopping early.")
            break
        for k in epoch_losses:
            epoch_losses[k] /= n_batches
        
        # Early stop on NaN
        if np.isnan(epoch_losses.get('loss', 0)):
            print(f"  WARNING: NaN loss at epoch {epoch}. Stopping early.")
            break
        
        # Validation
        model.eval()
        val_losses = defaultdict(float)
        n_val = 0
        with torch.no_grad():
            for data, factors in val_loader:
                data = data.to(device)
                recon, mu, logvar, z = model(data)
                losses = model.loss_function(data, recon, mu, logvar, z=z, dataset_size=dataset_size)
                for k, v in losses.items():
                    val_losses[k] += v.item()
                n_val += 1
        for k in val_losses:
            val_losses[k] /= max(n_val, 1)
        
        # KL per dimension
        kl_per_dim = []
        with torch.no_grad():
            all_mu, all_logvar = [], []
            for data, _ in val_loader:
                data = data.to(device)
                mu_v, logvar_v = model.encode(data)
                all_mu.append(mu_v)
                all_logvar.append(logvar_v)
            all_mu = torch.cat(all_mu, dim=0)
            all_logvar = torch.cat(all_logvar, dim=0)
            kl_per_dim = (-0.5 * (1 + all_logvar - all_mu.pow(2) - all_logvar.exp())).mean(dim=0).cpu().numpy()
        
        history['train_loss'].append(epoch_losses['loss'])
        history['val_loss'].append(val_losses['loss'])
        history['kl_per_dim'].append(kl_per_dim.tolist())
        
        # Log to W&B
        log_dict = {
            'epoch': epoch,
            'train/loss': epoch_losses['loss'],
            'train/recon_loss': epoch_losses.get('recon_loss', 0),
            'train/kl_loss': epoch_losses.get('kl_loss', 0),
            'val/loss': val_losses['loss'],
            'val/recon_loss': val_losses.get('recon_loss', 0),
            'val/kl_loss': val_losses.get('kl_loss', 0),
            'lr': scheduler.get_last_lr()[0],
        }
        for d_i, kl_val in enumerate(kl_per_dim):
            log_dict[f'kl_dim/z{d_i}'] = kl_val
        
        if isinstance(model, FactorVAE):
            log_dict['train/tc_loss'] = epoch_losses.get('tc_loss', 0)
        elif isinstance(model, BetaTCVAE):
            log_dict['train/mi'] = epoch_losses.get('mi', 0)
            log_dict['train/tc'] = epoch_losses.get('tc', 0)
            log_dict['train/dimwise_kl'] = epoch_losses.get('dimwise_kl', 0)
        elif isinstance(model, DIPVAE):
            log_dict['train/dip_loss'] = epoch_losses.get('dip_loss', 0)
        
        log_metrics(log_dict, prefix=f"{model_name}/{dataset_name}/seed{seed}")
        
        # Save checkpoint
        if val_losses['loss'] < best_val_loss:
            best_val_loss = val_losses['loss']
            save_checkpoint(model, optimizer, epoch, best_val_loss, ckpt_path,
                          extra={'history': history, 'model_name': model_name,
                                 'dataset_name': dataset_name, 'seed': seed})
        
        if epoch % max(1, epochs // 5) == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch}/{epochs-1} | Train: {epoch_losses['loss']:.4f} | "
                  f"Val: {val_losses['loss']:.4f} | Best: {best_val_loss:.4f}")
    
    # Load best model
    load_checkpoint(model, ckpt_path, device=device)
    
    return model, history


def prepare_data_loaders(dataset, batch_size=BATCH_SIZE, val_split=0.1):
    """Create train/val data loaders."""
    n = len(dataset)
    n_val = int(n * val_split)
    n_train = n - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val])
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=4, pin_memory=True)
    
    return train_loader, val_loader

print("Training infrastructure ready.")

## 7. Causal Intervention Framework

Four levels of interventions:
1. **Input Interventions** — modify input factors, track propagation
2. **Latent Space Interventions** — perturb individual z dimensions
3. **Activation Patching** — swap activations between inputs
4. **Causal Mediation Analysis** — identify mediating components

In [ ]:
class CausalInterventionFramework:
    """Complete 4-level causal intervention framework for VAEs."""
    
    def __init__(self, model, dataset, device=DEVICE):
        self.model = model.to(device).eval()
        self.dataset = dataset
        self.device = device
        self.factor_names = dataset.factor_names
        self.num_factors = dataset.num_factors
    
    # ================================================================
    # LEVEL 1: Input Interventions
    # ================================================================
    def input_interventions(self, n_samples=200):
        """Modify input factors and track propagation through network."""
        results = {f: {'delta_z': [], 'delta_recon': [], 'layer_responses': {}} 
                   for f in self.factor_names}
        
        loader = DataLoader(self.dataset, batch_size=1, shuffle=True)
        
        with torch.no_grad():
            for i, (x, factors) in enumerate(loader):
                if i >= n_samples:
                    break
                x = x.to(self.device)
                
                # Get baseline activations
                base_acts = self.model.get_all_activations(x)
                base_mu = base_acts['mu']
                
                # For each factor, create a perturbed version
                for f_idx, f_name in enumerate(self.factor_names):
                    # Create perturbed input by modifying the factor value
                    # Find a sample with different factor value
                    factor_vals = self.dataset.factors[:, f_idx]
                    unique_vals = np.unique(factor_vals)
                    if len(unique_vals) < 2:
                        continue
                    
                    current_val = factors[0, f_idx].item()
                    # Pick a different value
                    other_vals = unique_vals[unique_vals != current_val]
                    if len(other_vals) == 0:
                        continue
                    target_val = np.random.choice(other_vals)
                    
                    # Find sample with target factor value (and similar other factors)
                    mask = np.abs(factor_vals - target_val) < 1e-6
                    candidates = np.where(mask)[0]
                    if len(candidates) == 0:
                        continue
                    
                    alt_idx = np.random.choice(candidates)
                    x_alt = torch.tensor(self.dataset.images[alt_idx:alt_idx+1]).to(self.device)
                    
                    alt_acts = self.model.get_all_activations(x_alt)
                    
                    # Track differences
                    delta_z = (alt_acts['mu'] - base_mu).cpu().numpy().flatten()
                    results[f_name]['delta_z'].append(delta_z)
                    
                    # Track layer-wise responses
                    for layer_name in base_acts:
                        if layer_name in ['mu', 'logvar', 'z']:
                            continue
                        if layer_name not in results[f_name]['layer_responses']:
                            results[f_name]['layer_responses'][layer_name] = []
                        delta = (alt_acts[layer_name] - base_acts[layer_name]).abs().mean().item()
                        results[f_name]['layer_responses'][layer_name].append(delta)
        
        # Aggregate
        for f_name in results:
            if len(results[f_name]['delta_z']) > 0:
                results[f_name]['delta_z_mean'] = np.mean(results[f_name]['delta_z'], axis=0)
                results[f_name]['delta_z_std'] = np.std(results[f_name]['delta_z'], axis=0)
            for layer_name in results[f_name]['layer_responses']:
                vals = results[f_name]['layer_responses'][layer_name]
                results[f_name]['layer_responses'][layer_name] = {
                    'mean': float(np.mean(vals)), 'std': float(np.std(vals))
                }
        
        return results
    
    # ================================================================
    # LEVEL 2: Latent Space Interventions
    # ================================================================
    def latent_interventions(self, n_samples=200, z_range=INTERVENTION_RANGE, n_steps=INTERVENTION_STEPS):
        """Perturb individual latent dimensions and measure reconstruction effects."""
        results = {
            'ces': np.zeros(self.model.latent_dim),       # Causal Effect Strength
            'specificity': np.zeros(self.model.latent_dim), # Intervention Specificity
            'ces_per_sample': [],
            'traversals': [],
        }
        
        loader = DataLoader(self.dataset, batch_size=min(n_samples, 64), shuffle=True)
        all_ces = []
        all_spec = []
        
        z_values = np.linspace(z_range[0], z_range[1], n_steps)
        
        with torch.no_grad():
            sample_count = 0
            for x, factors in loader:
                if sample_count >= n_samples:
                    break
                x = x.to(self.device)
                mu, logvar = self.model.encode(x)
                z = mu  # Use mean for deterministic interventions
                recon_base = self.model.decode(z)
                
                batch_ces = np.zeros((x.size(0), self.model.latent_dim))
                batch_spec = np.zeros((x.size(0), self.model.latent_dim))
                
                for dim in range(self.model.latent_dim):
                    z_mod = z.clone()
                    # Sweep across range
                    total_effect = torch.zeros_like(recon_base)
                    for val in z_values:
                        z_mod[:, dim] = val
                        recon_mod = self.model.decode(z_mod)
                        total_effect += (recon_mod - recon_base).pow(2)
                    total_effect /= n_steps
                    
                    # CES: average L2 effect
                    ces_vals = total_effect.reshape(x.size(0), -1).sum(dim=1).sqrt().cpu().numpy()
                    batch_ces[:, dim] = ces_vals
                    
                    # Specificity: inverse entropy of pixel-wise effects
                    for b in range(x.size(0)):
                        pixel_effects = total_effect[b].reshape(-1).cpu().numpy()
                        pixel_effects = pixel_effects / (pixel_effects.sum() + 1e-10)
                        entropy = entr(pixel_effects).sum()
                        batch_spec[b, dim] = 1.0 / (entropy + 1e-6)
                
                all_ces.append(batch_ces)
                all_spec.append(batch_spec)
                sample_count += x.size(0)
            
            # Save some traversals for visualization
            x_sample = next(iter(DataLoader(self.dataset, batch_size=4, shuffle=True)))[0].to(self.device)
            mu_s, _ = self.model.encode(x_sample)
            for dim in range(self.model.latent_dim):
                trav = []
                for val in z_values:
                    z_t = mu_s.clone()
                    z_t[:, dim] = val
                    trav.append(self.model.decode(z_t).cpu())
                results['traversals'].append(torch.stack(trav, dim=1))
        
        all_ces = np.concatenate(all_ces, axis=0)
        all_spec = np.concatenate(all_spec, axis=0)
        
        results['ces'] = all_ces.mean(axis=0)
        results['ces_std'] = all_ces.std(axis=0)
        results['specificity'] = all_spec.mean(axis=0)
        results['specificity_std'] = all_spec.std(axis=0)
        results['ces_per_sample'] = all_ces
        
        return results
    
    # ================================================================
    # LEVEL 3: Activation Patching
    # ================================================================
    def activation_patching(self, n_pairs=100):
        """Patch activations between input pairs to identify causal pathways."""
        results = {
            'patch_effects': {},  # layer -> (n_pairs, n_neurons) effect matrix
            'factor_patch_effects': {f: {} for f in self.factor_names},
        }
        
        loader = DataLoader(self.dataset, batch_size=2, shuffle=True)
        
        with torch.no_grad():
            pair_count = 0
            for (x_pair, factors_pair) in loader:
                if pair_count >= n_pairs:
                    break
                if x_pair.size(0) < 2:
                    continue
                
                x1 = x_pair[0:1].to(self.device)
                x2 = x_pair[1:2].to(self.device)
                f1 = factors_pair[0].numpy()
                f2 = factors_pair[1].numpy()
                
                acts1 = self.model.get_all_activations(x1)
                acts2 = self.model.get_all_activations(x2)
                recon1 = self.model.decode(acts1['z'])
                
                # For each layer, patch activations and measure effect
                for layer_name in acts1:
                    if layer_name in ['z', 'logvar']:
                        continue
                    
                    act1 = acts1[layer_name]
                    act2 = acts2[layer_name]
                    
                    if act1.dim() == 2:
                        n_units = act1.size(1)
                    else:
                        n_units = act1.size(1)  # channels
                    
                    if layer_name not in results['patch_effects']:
                        results['patch_effects'][layer_name] = []
                    
                    unit_effects = np.zeros(n_units)
                    
                    for u in range(min(n_units, 32)):  # Limit for efficiency
                        patched = act1.clone()
                        if patched.dim() == 2:
                            patched[0, u] = act2[0, u]
                        else:
                            patched[0, u] = act2[0, u]
                        
                        # Forward from patched activation
                        if 'encoder' in layer_name or layer_name == 'mu':
                            # Need to forward through rest of encoder + decoder
                            if layer_name == 'mu':
                                z_patched = patched
                                recon_patched = self.model.decode(z_patched)
                            else:
                                continue  # Skip intermediate encoder layers for simplicity
                        else:
                            continue
                        
                        effect = (recon_patched - recon1).pow(2).mean().item()
                        unit_effects[u] = effect
                    
                    results['patch_effects'][layer_name].append(unit_effects)
                    
                    # Track which factors differ
                    for f_idx, f_name in enumerate(self.factor_names):
                        if abs(f1[f_idx] - f2[f_idx]) > 1e-6:
                            if layer_name not in results['factor_patch_effects'][f_name]:
                                results['factor_patch_effects'][f_name][layer_name] = []
                            results['factor_patch_effects'][f_name][layer_name].append(unit_effects)
                
                pair_count += 1
        
        # Aggregate
        for layer_name in results['patch_effects']:
            arr = np.array(results['patch_effects'][layer_name])
            results['patch_effects'][layer_name] = {
                'mean': arr.mean(axis=0), 'std': arr.std(axis=0)
            }
        
        for f_name in results['factor_patch_effects']:
            for layer_name in results['factor_patch_effects'][f_name]:
                arr = np.array(results['factor_patch_effects'][f_name][layer_name])
                if len(arr) > 0:
                    results['factor_patch_effects'][f_name][layer_name] = {
                        'mean': arr.mean(axis=0), 'std': arr.std(axis=0)
                    }
        
        return results
    
    # ================================================================
    # LEVEL 4: Causal Mediation Analysis
    # ================================================================
    def causal_mediation_analysis(self, n_samples=200):
        """Quantify mediation effects of each layer/component."""
        results = {f: {} for f in self.factor_names}
        
        loader = DataLoader(self.dataset, batch_size=2, shuffle=True)
        
        with torch.no_grad():
            sample_count = 0
            for x_pair, factors_pair in loader:
                if sample_count >= n_samples:
                    break
                if x_pair.size(0) < 2:
                    continue
                
                x1 = x_pair[0:1].to(self.device)
                x2 = x_pair[1:2].to(self.device)
                f1 = factors_pair[0].numpy()
                f2 = factors_pair[1].numpy()
                
                # Total effect
                mu1, _ = self.model.encode(x1)
                mu2, _ = self.model.encode(x2)
                total_effect = (mu2 - mu1).pow(2).sum().item()
                
                if total_effect < 1e-8:
                    continue
                
                # Get all activations
                acts1 = self.model.get_all_activations(x1)
                acts2 = self.model.get_all_activations(x2)
                
                # For each factor that differs
                for f_idx, f_name in enumerate(self.factor_names):
                    if abs(f1[f_idx] - f2[f_idx]) < 1e-6:
                        continue
                    
                    # Mediation through each encoder layer
                    for layer_name in ['encoder_conv_0', 'encoder_conv_1', 'encoder_conv_2']:
                        if layer_name not in acts1:
                            continue
                        
                        # Compute mediation: how much of total effect goes through this layer?
                        act_diff = (acts2[layer_name] - acts1[layer_name]).pow(2).mean().item()
                        mediation_strength = act_diff / (total_effect + 1e-8)
                        
                        if layer_name not in results[f_name]:
                            results[f_name][layer_name] = []
                        results[f_name][layer_name].append(min(mediation_strength, 10.0))
                
                sample_count += 1
        
        # Aggregate
        summary = {}
        for f_name in results:
            summary[f_name] = {}
            for layer_name in results[f_name]:
                vals = results[f_name][layer_name]
                if len(vals) > 0:
                    summary[f_name][layer_name] = {
                        'mean': float(np.mean(vals)),
                        'std': float(np.std(vals)),
                        'n': len(vals)
                    }
        
        return summary
    
    # ================================================================
    # Run all interventions
    # ================================================================
    def run_all(self, n_samples=None):
        """Run the complete intervention framework."""
        if n_samples is None:
            n_samples = NUM_INTERVENTION_SAMPLES
        
        print("  Level 1: Input interventions...")
        input_results = self.input_interventions(n_samples=n_samples)
        
        print("  Level 2: Latent space interventions...")
        latent_results = self.latent_interventions(n_samples=n_samples)
        
        print("  Level 3: Activation patching...")
        patch_results = self.activation_patching(n_pairs=min(n_samples, 200))
        
        print("  Level 4: Causal mediation analysis...")
        mediation_results = self.causal_mediation_analysis(n_samples=n_samples)
        
        return {
            'input': input_results,
            'latent': latent_results,
            'patching': patch_results,
            'mediation': mediation_results,
        }

print("Causal Intervention Framework ready.")

## 8. Disentanglement & Causal Metrics

Standard metrics: DCI, MIG, SAP, β-metric, FactorVAE metric  
Novel metrics: CES, Specificity, Modularity, Polysemanticity Score

In [ ]:
class DisentanglementMetrics:
    """Compute standard and novel disentanglement metrics."""
    
    def __init__(self, model, dataset, device=DEVICE, n_samples=None):
        self.model = model.to(device).eval()
        self.dataset = dataset
        self.device = device
        self.n_samples = n_samples or min(len(dataset), MAX_EVAL_SAMPLES)
    
    def _get_representations(self):
        """Extract latent representations for all samples."""
        n = min(self.n_samples, len(self.dataset))
        indices = np.random.choice(len(self.dataset), n, replace=False)
        
        all_z = []
        all_factors = []
        
        with torch.no_grad():
            for i in range(0, n, BATCH_SIZE):
                batch_idx = indices[i:i+BATCH_SIZE]
                imgs = torch.stack([torch.tensor(self.dataset.images[j]) for j in batch_idx]).to(self.device)
                factors = np.stack([self.dataset.factors[j] for j in batch_idx])
                
                mu, _ = self.model.encode(imgs)
                all_z.append(mu.cpu().numpy())
                all_factors.append(factors)
        
        return np.concatenate(all_z, axis=0), np.concatenate(all_factors, axis=0)
    
    def compute_dci(self):
        """DCI: Disentanglement, Completeness, Informativeness."""
        z, factors = self._get_representations()
        
        n_factors = factors.shape[1]
        n_codes = z.shape[1]
        
        # Importance matrix R: how important is code j for predicting factor k?
        R = np.zeros((n_codes, n_factors))
        informativeness = np.zeros(n_factors)
        
        for k in range(n_factors):
            # Discretize continuous factors
            y = factors[:, k]
            unique_vals = np.unique(y)
            
            if len(unique_vals) <= 10:
                # Classification
                model = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
                try:
                    model.fit(z, y.astype(int))
                    R[:, k] = model.feature_importances_
                    informativeness[k] = model.score(z, y.astype(int))
                except Exception:
                    pass
            else:
                # Regression
                model = GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42)
                try:
                    model.fit(z, y)
                    R[:, k] = model.feature_importances_
                    informativeness[k] = model.score(z, y)
                except Exception:
                    pass
        
        # Disentanglement: each code should be important for at most one factor
        R_normalized = R / (R.sum(axis=1, keepdims=True) + 1e-10)
        disentanglement_per_code = 1 - entr(R_normalized).sum(axis=1) / np.log(n_factors + 1e-10)
        rho = R.sum(axis=1) / (R.sum() + 1e-10)
        disentanglement = (rho * disentanglement_per_code).sum()
        
        # Completeness: each factor should be predicted by at most one code
        R_normalized_c = R / (R.sum(axis=0, keepdims=True) + 1e-10)
        completeness_per_factor = 1 - entr(R_normalized_c).sum(axis=0) / np.log(n_codes + 1e-10)
        completeness = completeness_per_factor.mean()
        
        return {
            'dci_disentanglement': float(disentanglement),
            'dci_completeness': float(completeness),
            'dci_informativeness': float(informativeness.mean()),
            'importance_matrix': R,
        }
    
    def compute_mig(self):
        """MIG: Mutual Information Gap."""
        z, factors = self._get_representations()
        
        n_factors = factors.shape[1]
        n_codes = z.shape[1]
        
        # Discretize z
        n_bins = 20
        z_discrete = np.zeros_like(z, dtype=int)
        for j in range(n_codes):
            bins = np.linspace(z[:, j].min() - 1e-6, z[:, j].max() + 1e-6, n_bins + 1)
            z_discrete[:, j] = np.digitize(z[:, j], bins) - 1
        
        # MI matrix
        mi_matrix = np.zeros((n_codes, n_factors))
        for j in range(n_codes):
            for k in range(n_factors):
                f_discrete = factors[:, k]
                unique_f = np.unique(f_discrete)
                if len(unique_f) > 20:
                    bins_f = np.linspace(f_discrete.min() - 1e-6, f_discrete.max() + 1e-6, 21)
                    f_discrete = np.digitize(f_discrete, bins_f) - 1
                
                mi_matrix[j, k] = mutual_info_score(z_discrete[:, j], f_discrete.astype(int))
        
        # Entropy of each code
        H_z = np.zeros(n_codes)
        for j in range(n_codes):
            vals, counts = np.unique(z_discrete[:, j], return_counts=True)
            H_z[j] = entr(counts / counts.sum()).sum()
        
        # MIG: for each factor, gap between top-2 MI values, normalized by H(z)
        mig_per_factor = np.zeros(n_factors)
        for k in range(n_factors):
            sorted_mi = np.sort(mi_matrix[:, k])[::-1]
            if len(sorted_mi) >= 2 and H_z[np.argmax(mi_matrix[:, k])] > 1e-10:
                j_max = np.argmax(mi_matrix[:, k])
                mig_per_factor[k] = (sorted_mi[0] - sorted_mi[1]) / (H_z[j_max] + 1e-10)
        
        return {
            'mig': float(mig_per_factor.mean()),
            'mig_per_factor': mig_per_factor.tolist(),
            'mi_matrix': mi_matrix,
        }
    
    def compute_sap(self):
        """SAP: Separated Attribute Predictability."""
        z, factors = self._get_representations()
        
        n_factors = factors.shape[1]
        n_codes = z.shape[1]
        
        # Score matrix: R^2 of linear regression from each code to each factor
        score_matrix = np.zeros((n_codes, n_factors))
        
        scaler = StandardScaler()
        z_scaled = scaler.fit_transform(z)
        
        for k in range(n_factors):
            y = factors[:, k]
            unique_vals = np.unique(y)
            
            for j in range(n_codes):
                if len(unique_vals) <= 10:
                    try:
                        clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
                        clf.fit(z_scaled[:, j:j+1], y.astype(int))
                        score_matrix[j, k] = clf.score(z_scaled[:, j:j+1], y.astype(int))
                    except Exception:
                        pass
                else:
                    corr = np.abs(np.corrcoef(z[:, j], y)[0, 1])
                    score_matrix[j, k] = corr if not np.isnan(corr) else 0
        
        # SAP: for each factor, gap between top-2 scores
        sap_per_factor = np.zeros(n_factors)
        for k in range(n_factors):
            sorted_scores = np.sort(score_matrix[:, k])[::-1]
            if len(sorted_scores) >= 2:
                sap_per_factor[k] = sorted_scores[0] - sorted_scores[1]
        
        return {
            'sap': float(sap_per_factor.mean()),
            'sap_per_factor': sap_per_factor.tolist(),
        }
    
    def compute_beta_metric(self):
        """β-metric: accuracy of linear classifier on latent traversals."""
        z, factors = self._get_representations()
        
        n_factors = factors.shape[1]
        
        # For each factor, find pairs that differ only in that factor
        n_train = min(1000, len(z) // 2)
        
        X_train = []
        y_train = []
        
        for _ in range(n_train):
            # Random factor
            k = np.random.randint(n_factors)
            
            # Find two samples that differ in factor k
            idx1 = np.random.randint(len(z))
            factor_vals = factors[:, k]  # Use subsampled factors, not self.dataset.factors
            current_val = factors[idx1, k]
            
            different = np.where(np.abs(factor_vals - current_val) > 1e-6)[0]
            if len(different) == 0:
                continue
            idx2 = np.random.choice(different)
            
            # Feature: which z dimension changed most
            z_diff = np.abs(z[idx1] - z[idx2])
            X_train.append(z_diff)
            y_train.append(k)
        
        if len(X_train) < 10:
            return {'beta_metric': 0.0}
        
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        
        clf = LogisticRegression(max_iter=1000, random_state=42)
        try:
            clf.fit(X_train, y_train)
            acc = clf.score(X_train, y_train)
        except Exception:
            acc = 0.0
        
        return {'beta_metric': float(acc)}
    
    def compute_factor_vae_metric(self):
        """FactorVAE metric: majority vote classifier on normalized variance."""
        z, factors = self._get_representations()
        
        n_factors = factors.shape[1]
        n_train = min(2000, len(z) // 2)
        
        votes = []
        true_labels = []
        
        for _ in range(n_train):
            k = np.random.randint(n_factors)
            
            # Get subset with fixed factor k
            factor_vals = factors[:, k]  # Use subsampled factors, not self.dataset.factors
            unique_vals = np.unique(factor_vals)
            if len(unique_vals) < 2:
                continue
            
            val = np.random.choice(unique_vals)
            mask = np.abs(factor_vals - val) < 1e-6
            subset_indices = np.where(mask)[0]
            
            if len(subset_indices) < 10:
                continue
            
            # Sample subset of z values with this fixed factor
            sample_idx = np.random.choice(subset_indices, min(50, len(subset_indices)), replace=False)
            z_subset = z[sample_idx]
            
            # Variance of each dimension
            var = z_subset.var(axis=0)
            normalized_var = var / (var.sum() + 1e-10)
            
            # Vote: dimension with lowest variance -> this factor
            vote = np.argmin(normalized_var)
            votes.append(vote)
            true_labels.append(k)
        
        if len(votes) == 0:
            return {'factor_vae_metric': 0.0}
        
        votes = np.array(votes)
        true_labels = np.array(true_labels)
        
        # Majority vote: for each z dimension, which factor does it vote for?
        dim_to_factor = {}
        for dim in range(z.shape[1]):
            dim_votes = true_labels[votes == dim]
            if len(dim_votes) > 0:
                vals, counts = np.unique(dim_votes, return_counts=True)
                dim_to_factor[dim] = vals[np.argmax(counts)]
        
        # Accuracy
        correct = sum(1 for v, t in zip(votes, true_labels) 
                      if v in dim_to_factor and dim_to_factor[v] == t)
        acc = correct / len(votes) if len(votes) > 0 else 0.0
        
        return {'factor_vae_metric': float(acc)}
    
    def compute_all_standard_metrics(self):
        """Compute all standard disentanglement metrics."""
        print("    Computing DCI...")
        dci = self.compute_dci()
        print("    Computing MIG...")
        mig = self.compute_mig()
        print("    Computing SAP...")
        sap = self.compute_sap()
        print("    Computing β-metric...")
        beta = self.compute_beta_metric()
        print("    Computing FactorVAE metric...")
        fvae = self.compute_factor_vae_metric()
        
        return {**dci, **mig, **sap, **beta, **fvae}


# ============================================================
# Novel Causal Metrics (from intervention results)
# ============================================================

def compute_circuit_modularity(model, dataset, n_samples=200, device=DEVICE):
    """Compute circuit modularity for each layer."""
    model = model.to(device).eval()
    factor_names = dataset.factor_names
    n_factors = len(factor_names)
    
    # Collect activation changes for each factor intervention
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    
    layer_factor_activations = defaultdict(lambda: defaultdict(list))
    
    with torch.no_grad():
        for i, (x, factors) in enumerate(loader):
            if i >= n_samples:
                break
            x = x.to(device)
            base_acts = model.get_all_activations(x)
            
            for f_idx, f_name in enumerate(factor_names):
                factor_vals = dataset.factors[:, f_idx]
                unique_vals = np.unique(factor_vals)
                if len(unique_vals) < 2:
                    continue
                
                current_val = factors[0, f_idx].item()
                other_vals = unique_vals[unique_vals != current_val]
                if len(other_vals) == 0:
                    continue
                target_val = np.random.choice(other_vals)
                
                mask = np.abs(factor_vals - target_val) < 1e-6
                candidates = np.where(mask)[0]
                if len(candidates) == 0:
                    continue
                
                alt_idx = np.random.choice(candidates)
                x_alt = torch.tensor(dataset.images[alt_idx:alt_idx+1]).to(device)
                alt_acts = model.get_all_activations(x_alt)
                
                for layer_name in base_acts:
                    if layer_name in ['z', 'logvar']:
                        continue
                    delta = (alt_acts[layer_name] - base_acts[layer_name]).abs()
                    if delta.dim() == 2:
                        delta_flat = delta[0].cpu().numpy()
                    else:
                        delta_flat = delta[0].mean(dim=[1, 2]).cpu().numpy()
                    layer_factor_activations[layer_name][f_name].append(delta_flat)
    
    # Compute modularity per layer
    modularity = {}
    for layer_name in layer_factor_activations:
        factor_deltas = {}
        for f_name in factor_names:
            if f_name in layer_factor_activations[layer_name] and len(layer_factor_activations[layer_name][f_name]) > 0:
                factor_deltas[f_name] = np.mean(layer_factor_activations[layer_name][f_name], axis=0)
        
        if len(factor_deltas) < 2:
            modularity[layer_name] = 1.0
            continue
        
        # Modularity = 1 - average absolute correlation between factor responses
        keys = list(factor_deltas.keys())
        correlations = []
        for i, j in combinations(range(len(keys)), 2):
            v1, v2 = factor_deltas[keys[i]], factor_deltas[keys[j]]
            min_len = min(len(v1), len(v2))
            if min_len > 1:
                r, _ = stats.pearsonr(v1[:min_len], v2[:min_len])
                if not np.isnan(r):
                    correlations.append(abs(r))
        
        if len(correlations) > 0:
            modularity[layer_name] = float(1 - np.mean(correlations))
        else:
            modularity[layer_name] = 1.0
    
    return modularity


def compute_polysemanticity(model, dataset, n_samples=200, device=DEVICE):
    """Compute polysemanticity scores for each unit in each layer."""
    model = model.to(device).eval()
    factor_names = dataset.factor_names
    n_factors = len(factor_names)
    
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    
    layer_factor_responses = defaultdict(lambda: defaultdict(list))
    
    with torch.no_grad():
        for i, (x, factors) in enumerate(loader):
            if i >= n_samples:
                break
            x = x.to(device)
            base_acts = model.get_all_activations(x)
            
            for f_idx, f_name in enumerate(factor_names):
                factor_vals = dataset.factors[:, f_idx]
                unique_vals = np.unique(factor_vals)
                if len(unique_vals) < 2:
                    continue
                
                current_val = factors[0, f_idx].item()
                other_vals = unique_vals[unique_vals != current_val]
                if len(other_vals) == 0:
                    continue
                
                alt_idx = np.random.choice(np.where(np.abs(factor_vals - np.random.choice(other_vals)) < 1e-6)[0])
                x_alt = torch.tensor(dataset.images[alt_idx:alt_idx+1]).to(device)
                alt_acts = model.get_all_activations(x_alt)
                
                for layer_name in base_acts:
                    if layer_name in ['z', 'logvar']:
                        continue
                    delta = (alt_acts[layer_name] - base_acts[layer_name]).abs()
                    if delta.dim() == 2:
                        response = delta[0].cpu().numpy()
                    else:
                        response = delta[0].mean(dim=[1, 2]).cpu().numpy()
                    layer_factor_responses[layer_name][f_name].append(response)
    
    # Compute PS scores
    ps_scores = {}
    for layer_name in layer_factor_responses:
        factor_responses = {}
        for f_name in factor_names:
            if f_name in layer_factor_responses[layer_name] and len(layer_factor_responses[layer_name][f_name]) > 0:
                factor_responses[f_name] = np.mean(layer_factor_responses[layer_name][f_name], axis=0)
        
        if len(factor_responses) == 0:
            continue
        
        n_units = len(list(factor_responses.values())[0])
        ps = np.ones(n_units)
        
        for u in range(n_units):
            R = np.array([factor_responses[f][u] for f in factor_responses if u < len(factor_responses[f])])
            if R.sum() > 1e-10:
                ps[u] = (R**2).sum() / (R.sum()**2) * len(R)
        
        ps_scores[layer_name] = {
            'scores': ps.tolist(),
            'mean': float(ps.mean()),
            'pct_monosemantic': float((ps < 1.5).mean() * 100),
        }
    
    return ps_scores

print("Metrics computation ready.")

## 9. Publication-Grade Visualizations

All figures saved locally (PNG + PDF) and logged to W&B.

In [ ]:
class PublicationVisualizer:
    """Generate publication-quality figures."""
    
    @staticmethod
    def plot_training_curves(histories, model_names, dataset_name):
        """Plot training & validation loss curves."""
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        
        for i, (name, hist) in enumerate(zip(model_names, histories)):
            color = PALETTE[i % len(PALETTE)]
            axes[0].plot(hist['train_loss'], label=name, color=color, linewidth=1.5)
            axes[1].plot(hist['val_loss'], label=name, color=color, linewidth=1.5)
        
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Training Loss')
        axes[0].set_title(f'Training Loss — {dataset_name}')
        axes[0].legend(framealpha=0.9)
        
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Validation Loss')
        axes[1].set_title(f'Validation Loss — {dataset_name}')
        axes[1].legend(framealpha=0.9)
        
        fig.tight_layout()
        return save_figure(fig, f"training_curves_{dataset_name}", subdirectory="training")
    
    @staticmethod
    def plot_kl_per_dimension(kl_data, model_names, dataset_name):
        """Plot KL divergence per latent dimension."""
        n_models = len(model_names)
        fig, axes = plt.subplots(1, n_models, figsize=(4*n_models, 4), sharey=True)
        if n_models == 1:
            axes = [axes]
        
        for i, (name, kl) in enumerate(zip(model_names, kl_data)):
            # Handle empty or missing KL data
            if isinstance(kl, list) and len(kl) > 0:
                kl_final = kl[-1] if isinstance(kl[-1], (list, np.ndarray)) else kl
            elif isinstance(kl, np.ndarray):
                kl_final = kl
            else:
                kl_final = [0] * LATENT_DIMS
            
            if isinstance(kl_final, (list, np.ndarray)) and len(kl_final) == 0:
                kl_final = [0] * LATENT_DIMS
            axes[i].bar(range(len(kl_final)), kl_final, color=PALETTE[i], alpha=0.8, edgecolor='black', linewidth=0.5)
            axes[i].set_xlabel('Latent Dimension')
            axes[i].set_title(name)
            if i == 0:
                axes[i].set_ylabel('KL Divergence')
        
        fig.suptitle(f'KL per Dimension — {dataset_name}', fontsize=14, y=1.02)
        fig.tight_layout()
        return save_figure(fig, f"kl_per_dim_{dataset_name}", subdirectory="training")
    
    @staticmethod
    def plot_ces_comparison(ces_data, model_names, dataset_name):
        """Plot Causal Effect Strength comparison across models."""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        n_dims = len(ces_data[0]['ces'])
        x = np.arange(n_dims)
        width = 0.8 / len(model_names)
        
        for i, (name, data) in enumerate(zip(model_names, ces_data)):
            offset = (i - len(model_names)/2 + 0.5) * width
            axes[0].bar(x + offset, data['ces'], width, 
                       yerr=data.get('ces_std', None), label=name,
                       color=PALETTE[i], alpha=0.85, edgecolor='black', linewidth=0.5,
                       capsize=2, error_kw={'linewidth': 0.8})
        
        axes[0].set_xlabel('Latent Dimension')
        axes[0].set_ylabel('Causal Effect Strength (CES)')
        axes[0].set_title(f'CES per Dimension — {dataset_name}')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels([f'z{i}' for i in range(n_dims)])
        axes[0].legend(framealpha=0.9)
        
        # Specificity
        for i, (name, data) in enumerate(zip(model_names, ces_data)):
            offset = (i - len(model_names)/2 + 0.5) * width
            axes[1].bar(x + offset, data['specificity'], width,
                       yerr=data.get('specificity_std', None), label=name,
                       color=PALETTE[i], alpha=0.85, edgecolor='black', linewidth=0.5,
                       capsize=2, error_kw={'linewidth': 0.8})
        
        axes[1].set_xlabel('Latent Dimension')
        axes[1].set_ylabel('Intervention Specificity')
        axes[1].set_title(f'Specificity per Dimension — {dataset_name}')
        axes[1].set_xticks(x)
        axes[1].set_xticklabels([f'z{i}' for i in range(n_dims)])
        axes[1].legend(framealpha=0.9)
        
        fig.tight_layout()
        return save_figure(fig, f"ces_specificity_{dataset_name}", subdirectory="interventions")
    
    @staticmethod
    def plot_modularity_comparison(modularity_data, model_names, dataset_name):
        """Plot circuit modularity across layers and models."""
        # Collect all layer names
        all_layers = set()
        for mod_dict in modularity_data:
            all_layers.update(mod_dict.keys())
        all_layers = sorted(all_layers)
        
        fig, ax = plt.subplots(figsize=(10, 5))
        
        x = np.arange(len(all_layers))
        width = 0.8 / len(model_names)
        
        for i, (name, mod_dict) in enumerate(zip(model_names, modularity_data)):
            vals = [mod_dict.get(l, 0) for l in all_layers]
            offset = (i - len(model_names)/2 + 0.5) * width
            ax.bar(x + offset, vals, width, label=name,
                   color=PALETTE[i], alpha=0.85, edgecolor='black', linewidth=0.5)
        
        ax.set_xlabel('Layer')
        ax.set_ylabel('Circuit Modularity')
        ax.set_title(f'Circuit Modularity — {dataset_name}')
        ax.set_xticks(x)
        ax.set_xticklabels(all_layers, rotation=30, ha='right')
        ax.legend(framealpha=0.9)
        
        fig.tight_layout()
        return save_figure(fig, f"modularity_{dataset_name}", subdirectory="modularity")
    
    @staticmethod
    def plot_polysemanticity_histograms(ps_data, model_names, dataset_name):
        """Plot polysemanticity score distributions."""
        # Find common layers
        common_layers = set(ps_data[0].keys())
        for d in ps_data[1:]:
            common_layers &= set(d.keys())
        common_layers = sorted(common_layers)[:4]  # Top 4 layers
        
        if not common_layers:
            return None
        
        fig, axes = plt.subplots(len(common_layers), len(model_names),
                                 figsize=(4*len(model_names), 3*len(common_layers)),
                                 squeeze=False)
        
        for row, layer in enumerate(common_layers):
            for col, (name, ps_dict) in enumerate(zip(model_names, ps_data)):
                if layer in ps_dict:
                    scores = ps_dict[layer]['scores']
                    axes[row, col].hist(scores, bins=20, color=PALETTE[col], alpha=0.8,
                                       edgecolor='black', linewidth=0.5)
                    axes[row, col].axvline(x=1.5, color='red', linestyle='--', linewidth=1, alpha=0.7)
                    axes[row, col].set_title(f'{name} — {layer}', fontsize=10)
                    pct = ps_dict[layer]['pct_monosemantic']
                    axes[row, col].text(0.95, 0.95, f'{pct:.0f}% mono',
                                       transform=axes[row, col].transAxes,
                                       ha='right', va='top', fontsize=9,
                                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
                
                if col == 0:
                    axes[row, col].set_ylabel('Count')
                if row == len(common_layers) - 1:
                    axes[row, col].set_xlabel('Polysemanticity Score')
        
        fig.suptitle(f'Polysemanticity Distributions — {dataset_name}', fontsize=14, y=1.01)
        fig.tight_layout()
        return save_figure(fig, f"polysemanticity_{dataset_name}", subdirectory="polysemanticity")
    
    @staticmethod
    def plot_mediation_heatmap(mediation_data, model_name, dataset_name):
        """Plot causal mediation heatmap."""
        factors = list(mediation_data.keys())
        layers = set()
        for f in factors:
            layers.update(mediation_data[f].keys())
        layers = sorted(layers)
        
        if not layers or not factors:
            return None
        
        matrix = np.zeros((len(factors), len(layers)))
        for i, f in enumerate(factors):
            for j, l in enumerate(layers):
                if l in mediation_data[f]:
                    matrix[i, j] = mediation_data[f][l]['mean']
        
        fig, ax = plt.subplots(figsize=(max(8, len(layers)*1.5), max(4, len(factors)*0.6)))
        
        sns.heatmap(matrix, annot=True, fmt='.3f', cmap='YlOrRd',
                    xticklabels=layers, yticklabels=factors, ax=ax,
                    cbar_kws={'label': 'Mediation Strength'})
        
        ax.set_title(f'Causal Mediation — {model_name} on {dataset_name}')
        ax.set_xlabel('Layer')
        ax.set_ylabel('Factor')
        
        fig.tight_layout()
        return save_figure(fig, f"mediation_{model_name}_{dataset_name}", subdirectory="mediation")
    
    @staticmethod
    def plot_latent_traversals(traversals, model_name, dataset_name, n_dims=5, n_samples=3):
        """Plot latent dimension traversals."""
        n_dims = min(n_dims, len(traversals))
        n_steps = traversals[0].size(1)
        step_indices = np.linspace(0, n_steps-1, min(7, n_steps)).astype(int)
        
        fig, axes = plt.subplots(n_dims, len(step_indices), 
                                 figsize=(1.5*len(step_indices), 1.5*n_dims))
        if n_dims == 1:
            axes = axes[np.newaxis, :]
        
        for dim in range(n_dims):
            trav = traversals[dim][0]  # First sample
            for j, step_idx in enumerate(step_indices):
                img = trav[step_idx].squeeze().numpy()
                if img.ndim == 3:
                    img = np.transpose(img, (1, 2, 0))
                axes[dim, j].imshow(img, cmap='gray' if img.ndim == 2 else None)
                axes[dim, j].axis('off')
                if j == 0:
                    axes[dim, j].set_ylabel(f'z{dim}', fontsize=10, rotation=0, labelpad=20)
        
        fig.suptitle(f'Latent Traversals — {model_name} on {dataset_name}', fontsize=13, y=1.02)
        fig.tight_layout()
        return save_figure(fig, f"traversals_{model_name}_{dataset_name}", subdirectory="traversals")
    
    @staticmethod
    def plot_disentanglement_comparison(metrics_all, model_names, dataset_name):
        """Plot comparison of all disentanglement metrics."""
        metric_keys = ['dci_disentanglement', 'mig', 'sap', 'beta_metric', 'factor_vae_metric']
        metric_labels = ['DCI', 'MIG', 'SAP', 'β-metric', 'FactorVAE']
        
        fig, ax = plt.subplots(figsize=(10, 5))
        
        x = np.arange(len(metric_keys))
        width = 0.8 / len(model_names)
        
        for i, (name, metrics) in enumerate(zip(model_names, metrics_all)):
            vals = [metrics.get(k, 0) for k in metric_keys]
            offset = (i - len(model_names)/2 + 0.5) * width
            ax.bar(x + offset, vals, width, label=name,
                   color=PALETTE[i], alpha=0.85, edgecolor='black', linewidth=0.5)
        
        ax.set_ylabel('Score')
        ax.set_title(f'Disentanglement Metrics — {dataset_name}')
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels)
        ax.legend(framealpha=0.9, loc='upper right')
        
        fig.tight_layout()
        return save_figure(fig, f"disentanglement_comparison_{dataset_name}", subdirectory="metrics")
    
    @staticmethod
    def plot_modularity_paradox(all_results):
        """Plot the modularity paradox: M vs CES vs disentanglement."""
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
        
        model_names = []
        modularity_vals = []
        ces_vals = []
        disent_vals = []
        m_ces_vals = []
        
        for key, result in all_results.items():
            if 'metrics' in result and 'modularity' in result and 'interventions' in result:
                model_names.append(key)
                mod = result['modularity'].get('mu', 0.5)
                ces = np.mean(result['interventions']['latent']['ces'])
                dis = result['metrics'].get('dci_disentanglement', 0)
                modularity_vals.append(mod)
                ces_vals.append(ces)
                disent_vals.append(dis)
                m_ces_vals.append(mod * ces)
        
        if len(model_names) < 2:
            plt.close(fig)
            return None
        
        colors = [PALETTE[i % len(PALETTE)] for i in range(len(model_names))]
        
        # Plot 1: Modularity vs Disentanglement
        axes[0].scatter(modularity_vals, disent_vals, c=colors, s=100, edgecolors='black', linewidth=0.8, zorder=5)
        for i, name in enumerate(model_names):
            axes[0].annotate(name, (modularity_vals[i], disent_vals[i]), 
                           textcoords="offset points", xytext=(5, 5), fontsize=8)
        axes[0].set_xlabel('Modularity (mu layer)')
        axes[0].set_ylabel('DCI Disentanglement')
        axes[0].set_title('Modularity vs Disentanglement')
        
        # Plot 2: CES vs Disentanglement
        axes[1].scatter(ces_vals, disent_vals, c=colors, s=100, edgecolors='black', linewidth=0.8, zorder=5)
        for i, name in enumerate(model_names):
            axes[1].annotate(name, (ces_vals[i], disent_vals[i]),
                           textcoords="offset points", xytext=(5, 5), fontsize=8)
        axes[1].set_xlabel('Mean CES')
        axes[1].set_ylabel('DCI Disentanglement')
        axes[1].set_title('Effect Strength vs Disentanglement')
        
        # Plot 3: M x CES vs Disentanglement
        axes[2].scatter(m_ces_vals, disent_vals, c=colors, s=100, edgecolors='black', linewidth=0.8, zorder=5)
        for i, name in enumerate(model_names):
            axes[2].annotate(name, (m_ces_vals[i], disent_vals[i]),
                           textcoords="offset points", xytext=(5, 5), fontsize=8)
        if len(m_ces_vals) > 2:
            r, p = stats.pearsonr(m_ces_vals, disent_vals)
            axes[2].set_title(f'M×CES vs Disentanglement (r={r:.3f}, p={p:.3f})')
        else:
            axes[2].set_title('M×CES vs Disentanglement')
        axes[2].set_xlabel('Modularity × CES')
        axes[2].set_ylabel('DCI Disentanglement')
        
        fig.suptitle('The Modularity Paradox', fontsize=14, y=1.03)
        fig.tight_layout()
        return save_figure(fig, "modularity_paradox", subdirectory="analysis")
    
    @staticmethod
    def plot_summary_table(all_results, dataset_name):
        """Create a summary table as a figure."""
        model_names = sorted(all_results.keys())
        
        metric_keys = ['dci_disentanglement', 'mig', 'sap', 'beta_metric', 'factor_vae_metric']
        metric_labels = ['DCI ↑', 'MIG ↑', 'SAP ↑', 'β-metric ↑', 'FVM ↑', 'Avg CES', 'Avg Spec.', 'Mu Mod.']
        
        rows = []
        for name in model_names:
            r = all_results[name]
            row = []
            for k in metric_keys:
                row.append(f"{r.get('metrics', {}).get(k, 0):.4f}")
            
            if 'interventions' in r and 'latent' in r['interventions']:
                row.append(f"{np.mean(r['interventions']['latent']['ces']):.3f}")
                row.append(f"{np.mean(r['interventions']['latent']['specificity']):.4f}")
            else:
                row.extend(['—', '—'])
            
            row.append(f"{r.get('modularity', {}).get('mu', 0):.4f}")
            rows.append(row)
        
        fig, ax = plt.subplots(figsize=(14, max(3, len(model_names) * 0.6 + 1.5)))
        ax.axis('off')
        
        table = ax.table(cellText=rows, colLabels=metric_labels, rowLabels=model_names,
                        cellLoc='center', loc='center')
        
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.auto_set_column_width(range(len(metric_labels)))
        table.scale(1, 1.5)
        
        # Style header
        for j in range(len(metric_labels)):
            table[(0, j)].set_facecolor('#2c3e50')
            table[(0, j)].set_text_props(color='white', fontweight='bold')
        
        for i in range(len(model_names)):
            table[(i+1, -1)].set_text_props(fontweight='bold')
        
        ax.set_title(f'Results Summary — {dataset_name}', fontsize=14, pad=20)
        fig.tight_layout()
        return save_figure(fig, f"summary_table_{dataset_name}", subdirectory="summary")

viz = PublicationVisualizer()
print("Visualization module ready.")

## 10. Main Experiment Pipeline

This cell orchestrates the complete experiment: training all models on all datasets across all seeds, running interventions, computing metrics, and generating visualizations.

In [ ]:
def run_single_experiment(model_name, dataset_name, dataset, seed, device=DEVICE):
    """Run a single model-dataset-seed experiment."""
    print(f"\n{'='*60}")
    print(f"  Model: {model_name} | Dataset: {dataset_name} | Seed: {seed}")
    print(f"{'='*60}")
    
    set_seed(seed)
    
    in_channels = dataset.images.shape[1]
    model = create_model(model_name, in_channels, LATENT_DIMS)
    
    # Check for existing checkpoint
    ckpt_path = get_checkpoint_path(model_name, dataset_name, seed)
    
    # Prepare data
    max_samples = MAX_TRAIN_SAMPLES if TEST_MODE else None
    if max_samples and max_samples < len(dataset):
        indices = np.random.choice(len(dataset), max_samples, replace=False)
        subset = Subset(dataset, indices)
        train_loader, val_loader = prepare_data_loaders(subset)
    else:
        train_loader, val_loader = prepare_data_loaders(dataset)
    
    # Train
    print(f"  Training for {TRAIN_EPOCHS} epochs...")
    model, history = train_vae(model, train_loader, val_loader, model_name, dataset_name, seed)
    
    # Compute metrics
    print(f"  Computing disentanglement metrics...")
    metrics_computer = DisentanglementMetrics(model, dataset, device, n_samples=MAX_EVAL_SAMPLES)
    std_metrics = metrics_computer.compute_all_standard_metrics()
    
    # Run interventions
    print(f"  Running causal interventions...")
    framework = CausalInterventionFramework(model, dataset, device)
    interventions = framework.run_all(n_samples=NUM_INTERVENTION_SAMPLES)
    
    # Compute novel metrics
    print(f"  Computing circuit modularity...")
    modularity = compute_circuit_modularity(model, dataset, n_samples=NUM_INTERVENTION_SAMPLES, device=device)
    
    print(f"  Computing polysemanticity...")
    polysemanticity = compute_polysemanticity(model, dataset, n_samples=NUM_INTERVENTION_SAMPLES, device=device)
    
    # Compile results
    result = {
        'model_name': model_name,
        'dataset_name': dataset_name,
        'seed': seed,
        'history': history,
        'metrics': std_metrics,
        'interventions': {
            'latent': {
                'ces': interventions['latent']['ces'].tolist(),
                'ces_std': interventions['latent']['ces_std'].tolist(),
                'specificity': interventions['latent']['specificity'].tolist(),
                'specificity_std': interventions['latent']['specificity_std'].tolist(),
            },
            'mediation': interventions['mediation'],
        },
        'modularity': modularity,
        'polysemanticity': polysemanticity,
        'traversals': interventions['latent'].get('traversals', []),
    }
    
    # Log summary metrics to W&B
    summary = {
        'dci': std_metrics.get('dci_disentanglement', 0),
        'mig': std_metrics.get('mig', 0),
        'sap': std_metrics.get('sap', 0),
        'beta_metric': std_metrics.get('beta_metric', 0),
        'factor_vae_metric': std_metrics.get('factor_vae_metric', 0),
        'avg_ces': float(np.mean(interventions['latent']['ces'])),
        'avg_specificity': float(np.mean(interventions['latent']['specificity'])),
        'mu_modularity': modularity.get('mu', 0),
    }
    log_metrics(summary, prefix=f"summary/{model_name}/{dataset_name}/seed{seed}")
    
    print(f"  Done! DCI={summary['dci']:.4f}, MIG={summary['mig']:.4f}, CES={summary['avg_ces']:.3f}")
    
    return result


def run_full_pipeline():
    """Run the complete experiment pipeline."""
    
    # Initialize W&B
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=f"{EXPERIMENT_NAME}_{'test' if TEST_MODE else 'full'}_{time.strftime('%Y%m%d_%H%M%S')}",
        config={
            'test_mode': TEST_MODE,
            'num_seeds': NUM_SEEDS,
            'train_epochs': TRAIN_EPOCHS,
            'batch_size': BATCH_SIZE,
            'latent_dims': LATENT_DIMS,
            'learning_rate': LEARNING_RATE,
            'datasets': DATASETS_TO_RUN,
            'models': MODELS_TO_RUN,
            'model_configs': MODEL_CONFIGS,
            'intervention_range': INTERVENTION_RANGE,
            'num_intervention_samples': NUM_INTERVENTION_SAMPLES,
        },
        tags=['test' if TEST_MODE else 'full', 'vae-mi'],
    )
    
    all_results = {}
    
    for ds_name in DATASETS_TO_RUN:
        print(f"\n{'#'*70}")
        print(f"# DATASET: {ds_name}")
        print(f"{'#'*70}")
        
        # Load dataset
        try:
            dataset = load_dataset(ds_name, max_samples=MAX_TRAIN_SAMPLES)
        except Exception as e:
            print(f"  ERROR loading {ds_name}: {e}")
            print(f"  Skipping this dataset.")
            continue
        
        dataset_results = {}
        
        for model_name in MODELS_TO_RUN:
            seed_results = []
            
            for seed in range(NUM_SEEDS):
                try:
                    result = run_single_experiment(model_name, ds_name, dataset, seed)
                    seed_results.append(result)
                except Exception as e:
                    print(f"  ERROR in {model_name}/{ds_name}/seed{seed}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue
            
            if not seed_results:
                continue
            
            # Aggregate across seeds
            agg = aggregate_seed_results(seed_results)
            dataset_results[model_name] = agg
        
        # Generate per-dataset visualizations
        if len(dataset_results) >= 2:
            try:
                generate_dataset_visualizations(dataset_results, ds_name)
            except Exception as e:
                print(f"  WARNING: Visualization failed for {ds_name}: {e}")
                import traceback
                traceback.print_exc()
        
        all_results[ds_name] = dataset_results
    
    # Generate cross-dataset analysis
    if len(all_results) > 0:
        try:
            generate_cross_dataset_analysis(all_results)
        except Exception as e:
            print(f"  WARNING: Cross-dataset analysis failed: {e}")
            import traceback
            traceback.print_exc()
    
    # Save final results
    save_results(all_results, f"all_results_{'test' if TEST_MODE else 'full'}")
    
    wandb.finish()
    print(f"\n{'='*70}")
    print("EXPERIMENT COMPLETE!")
    print(f"  Results: {RESULTS_DIR}")
    print(f"  Figures: {FIGURES_DIR}")
    print(f"  Checkpoints: {CHECKPOINT_DIR}")
    print(f"{'='*70}")
    
    return all_results


def aggregate_seed_results(seed_results):
    """Aggregate results across multiple seeds, computing mean and std."""
    agg = {
        'model_name': seed_results[0]['model_name'],
        'dataset_name': seed_results[0]['dataset_name'],
        'n_seeds': len(seed_results),
    }
    
    # Aggregate standard metrics
    metric_keys = ['dci_disentanglement', 'dci_completeness', 'dci_informativeness',
                   'mig', 'sap', 'beta_metric', 'factor_vae_metric']
    
    agg['metrics'] = {}
    for key in metric_keys:
        vals = [r['metrics'].get(key, 0) for r in seed_results]
        agg['metrics'][key] = float(np.mean(vals))
        agg['metrics'][f'{key}_std'] = float(np.std(vals))
    
    # Aggregate CES
    ces_arrays = [np.array(r['interventions']['latent']['ces']) for r in seed_results]
    spec_arrays = [np.array(r['interventions']['latent']['specificity']) for r in seed_results]
    
    agg['interventions'] = {
        'latent': {
            'ces': np.mean(ces_arrays, axis=0).tolist(),
            'ces_std': np.std(ces_arrays, axis=0).tolist(),
            'specificity': np.mean(spec_arrays, axis=0).tolist(),
            'specificity_std': np.std(spec_arrays, axis=0).tolist(),
        },
        'mediation': seed_results[0]['interventions']['mediation'],
    }
    
    # Aggregate modularity
    mod_keys = set()
    for r in seed_results:
        mod_keys.update(r['modularity'].keys())
    
    agg['modularity'] = {}
    for key in mod_keys:
        vals = [r['modularity'].get(key, 0) for r in seed_results]
        agg['modularity'][key] = float(np.mean(vals))
    
    # Polysemanticity from first seed (representative)
    agg['polysemanticity'] = seed_results[0]['polysemanticity']
    
    # Keep traversals from first seed
    agg['traversals'] = seed_results[0].get('traversals', [])
    
    # History from first seed
    agg['history'] = seed_results[0]['history']
    
    return agg


def generate_dataset_visualizations(dataset_results, dataset_name):
    """Generate all visualizations for a single dataset."""
    print(f"\n  Generating visualizations for {dataset_name}...")
    
    model_names = sorted(dataset_results.keys())
    
    # Training curves
    histories = [dataset_results[m]['history'] for m in model_names]
    viz.plot_training_curves(histories, model_names, dataset_name)
    
    # KL per dimension
    kl_data = [h.get('kl_per_dim', [[0]*LATENT_DIMS]) for h in histories]
    viz.plot_kl_per_dimension(kl_data, model_names, dataset_name)
    
    # CES comparison
    ces_data = [dataset_results[m]['interventions']['latent'] for m in model_names]
    viz.plot_ces_comparison(ces_data, model_names, dataset_name)
    
    # Modularity
    mod_data = [dataset_results[m]['modularity'] for m in model_names]
    viz.plot_modularity_comparison(mod_data, model_names, dataset_name)
    
    # Polysemanticity
    ps_data = [dataset_results[m]['polysemanticity'] for m in model_names]
    viz.plot_polysemanticity_histograms(ps_data, model_names, dataset_name)
    
    # Mediation heatmaps (per model)
    for m in model_names:
        mediation = dataset_results[m]['interventions'].get('mediation', {})
        if mediation:
            viz.plot_mediation_heatmap(mediation, m, dataset_name)
    
    # Latent traversals (per model)
    for m in model_names:
        traversals = dataset_results[m].get('traversals', [])
        if traversals:
            viz.plot_latent_traversals(traversals, m, dataset_name)
    
    # Disentanglement comparison
    metrics_all = [dataset_results[m]['metrics'] for m in model_names]
    viz.plot_disentanglement_comparison(metrics_all, model_names, dataset_name)
    
    # Summary table
    viz.plot_summary_table(dataset_results, dataset_name)
    
    # Modularity paradox
    viz.plot_modularity_paradox(dataset_results)
    
    print(f"  Visualizations saved to {FIGURES_DIR}")


def generate_cross_dataset_analysis(all_results):
    """Generate cross-dataset comparison figures."""
    print("\n  Generating cross-dataset analysis...")
    
    # Collect per-model averages across datasets
    model_dataset_metrics = defaultdict(lambda: defaultdict(list))
    
    for ds_name, ds_results in all_results.items():
        for m_name, result in ds_results.items():
            for metric_key in ['dci_disentanglement', 'mig', 'sap']:
                val = result.get('metrics', {}).get(metric_key, 0)
                model_dataset_metrics[m_name][metric_key].append(val)
            
            ces = result.get('interventions', {}).get('latent', {}).get('ces', [])
            if ces:
                model_dataset_metrics[m_name]['avg_ces'].append(float(np.mean(ces)))
            
            mod = result.get('modularity', {}).get('mu', 0)
            model_dataset_metrics[m_name]['mu_modularity'].append(mod)
    
    # Plot: model ranking across datasets
    model_names = sorted(model_dataset_metrics.keys())
    metrics_to_plot = ['dci_disentanglement', 'mig', 'sap', 'avg_ces', 'mu_modularity']
    labels = ['DCI', 'MIG', 'SAP', 'Avg CES', 'Mu Modularity']
    
    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4*len(metrics_to_plot), 5))
    
    for j, (metric, label) in enumerate(zip(metrics_to_plot, labels)):
        means = []
        stds = []
        for m in model_names:
            vals = model_dataset_metrics[m].get(metric, [0])
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        
        x = np.arange(len(model_names))
        axes[j].bar(x, means, yerr=stds, color=[PALETTE[i] for i in range(len(model_names))],
                   alpha=0.85, edgecolor='black', linewidth=0.5, capsize=3)
        axes[j].set_xticks(x)
        axes[j].set_xticklabels(model_names, rotation=45, ha='right', fontsize=9)
        axes[j].set_ylabel(label)
        axes[j].set_title(label)
    
    fig.suptitle('Cross-Dataset Model Comparison', fontsize=14, y=1.03)
    fig.tight_layout()
    save_figure(fig, "cross_dataset_comparison", subdirectory="summary")
    
    # Save cross-dataset summary
    summary = {}
    for m in model_names:
        summary[m] = {
            metric: {
                'mean': float(np.mean(model_dataset_metrics[m].get(metric, [0]))),
                'std': float(np.std(model_dataset_metrics[m].get(metric, [0]))),
                'per_dataset': model_dataset_metrics[m].get(metric, []),
            }
            for metric in metrics_to_plot
        }
    save_results(summary, "cross_dataset_summary")

print("Experiment pipeline ready. Call run_full_pipeline() to start.")

## 11. Run the Experiment

Execute the full pipeline. In **TEST_MODE**, this runs a quick validation pass.
Set `TEST_MODE = False` above and re-run for the full experiment.

In [ ]:
# Run the experiment!
all_results = run_full_pipeline()

## 12. Post-hoc Analysis & Summary

After the main experiment completes, run additional analyses.

In [ ]:
def print_summary(all_results):
    """Print a comprehensive text summary."""
    print("\n" + "=" * 80)
    print("EXPERIMENT SUMMARY")
    print("=" * 80)
    
    for ds_name, ds_results in all_results.items():
        print(f"\n{'─'*60}")
        print(f"Dataset: {ds_name}")
        print(f"{'─'*60}")
        
        for m_name in sorted(ds_results.keys()):
            r = ds_results[m_name]
            metrics = r.get('metrics', {})
            ces = r.get('interventions', {}).get('latent', {}).get('ces', [])
            mod = r.get('modularity', {}).get('mu', 0)
            
            print(f"\n  {m_name}:")
            print(f"    DCI Disentanglement: {metrics.get('dci_disentanglement', 0):.4f} "
                  f"(±{metrics.get('dci_disentanglement_std', 0):.4f})")
            print(f"    MIG:                 {metrics.get('mig', 0):.4f} "
                  f"(±{metrics.get('mig_std', 0):.4f})")
            print(f"    SAP:                 {metrics.get('sap', 0):.4f}")
            print(f"    β-metric:            {metrics.get('beta_metric', 0):.4f}")
            print(f"    FactorVAE metric:    {metrics.get('factor_vae_metric', 0):.4f}")
            if ces:
                print(f"    Avg CES:             {np.mean(ces):.4f}")
            print(f"    Mu Modularity:       {mod:.4f}")
    
    print(f"\n{'='*80}")
    print(f"All results saved to: {RESULTS_DIR}")
    print(f"All figures saved to: {FIGURES_DIR}")
    print(f"All checkpoints at:   {CHECKPOINT_DIR}")
    print(f"{'='*80}")

print_summary(all_results)

In [ ]:
# Beta/Gamma sweep for modularity paradox (run only in full mode)
if not TEST_MODE and len(all_results) > 0:
    print("\nRunning beta/gamma sweep for modularity paradox analysis...")
    
    ds_name = DATASETS_TO_RUN[0]
    dataset = load_dataset(ds_name, max_samples=MAX_TRAIN_SAMPLES)
    in_channels = dataset.images.shape[1]
    
    sweep_results = {}
    
    # Beta sweep
    for beta in BETA_SWEEP:
        name = f"beta_vae_b{beta}"
        print(f"  Training {name}...")
        set_seed(0)
        model = BetaVAE(in_channels, LATENT_DIMS, beta=beta).to(DEVICE)
        train_loader, val_loader = prepare_data_loaders(dataset)
        model, history = train_vae(model, train_loader, val_loader, name, ds_name, 0)
        
        metrics_comp = DisentanglementMetrics(model, dataset, DEVICE, n_samples=MAX_EVAL_SAMPLES)
        std_metrics = metrics_comp.compute_all_standard_metrics()
        
        framework = CausalInterventionFramework(model, dataset, DEVICE)
        latent_results = framework.latent_interventions(n_samples=NUM_INTERVENTION_SAMPLES)
        modularity = compute_circuit_modularity(model, dataset, n_samples=NUM_INTERVENTION_SAMPLES)
        
        sweep_results[name] = {
            'beta': beta,
            'dci': std_metrics.get('dci_disentanglement', 0),
            'avg_ces': float(np.mean(latent_results['ces'])),
            'mu_modularity': modularity.get('mu', 0),
            'm_x_ces': modularity.get('mu', 0) * float(np.mean(latent_results['ces'])),
        }
    
    # Gamma sweep
    for gamma in GAMMA_SWEEP:
        name = f"factor_vae_g{gamma}"
        print(f"  Training {name}...")
        set_seed(0)
        model = FactorVAE(in_channels, LATENT_DIMS, gamma=gamma).to(DEVICE)
        train_loader, val_loader = prepare_data_loaders(dataset)
        model, history = train_vae(model, train_loader, val_loader, name, ds_name, 0)
        
        metrics_comp = DisentanglementMetrics(model, dataset, DEVICE, n_samples=MAX_EVAL_SAMPLES)
        std_metrics = metrics_comp.compute_all_standard_metrics()
        
        framework = CausalInterventionFramework(model, dataset, DEVICE)
        latent_results = framework.latent_interventions(n_samples=NUM_INTERVENTION_SAMPLES)
        modularity = compute_circuit_modularity(model, dataset, n_samples=NUM_INTERVENTION_SAMPLES)
        
        sweep_results[name] = {
            'gamma': gamma,
            'dci': std_metrics.get('dci_disentanglement', 0),
            'avg_ces': float(np.mean(latent_results['ces'])),
            'mu_modularity': modularity.get('mu', 0),
            'm_x_ces': modularity.get('mu', 0) * float(np.mean(latent_results['ces'])),
        }
    
    # Visualize sweep
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pareto frontier: Modularity vs CES, colored by DCI
    names = list(sweep_results.keys())
    mods = [sweep_results[n]['mu_modularity'] for n in names]
    ces_vals = [sweep_results[n]['avg_ces'] for n in names]
    dcis = [sweep_results[n]['dci'] for n in names]
    
    sc = axes[0].scatter(mods, ces_vals, c=dcis, s=100, cmap='viridis',
                         edgecolors='black', linewidth=0.8)
    for i, n in enumerate(names):
        axes[0].annotate(n.replace('_', '\n'), (mods[i], ces_vals[i]),
                        textcoords="offset points", xytext=(5, 5), fontsize=7)
    plt.colorbar(sc, ax=axes[0], label='DCI Disentanglement')
    axes[0].set_xlabel('Mu Modularity')
    axes[0].set_ylabel('Average CES')
    axes[0].set_title('Modularity vs CES (colored by DCI)')
    
    # M x CES vs DCI
    m_ces = [sweep_results[n]['m_x_ces'] for n in names]
    axes[1].scatter(m_ces, dcis, s=100, c=PALETTE[:len(names)],
                   edgecolors='black', linewidth=0.8)
    for i, n in enumerate(names):
        axes[1].annotate(n.replace('_', '\n'), (m_ces[i], dcis[i]),
                        textcoords="offset points", xytext=(5, 5), fontsize=7)
    
    if len(m_ces) > 2:
        r, p = stats.pearsonr(m_ces, dcis)
        axes[1].set_title(f'M×CES vs DCI (r={r:.3f}, p={p:.3f})')
    else:
        axes[1].set_title('M×CES vs DCI')
    axes[1].set_xlabel('Modularity × CES')
    axes[1].set_ylabel('DCI Disentanglement')
    
    fig.suptitle(f'Hyperparameter Sweep — {ds_name}', fontsize=14, y=1.03)
    fig.tight_layout()
    save_figure(fig, f"hp_sweep_{ds_name}", subdirectory="analysis")
    save_results(sweep_results, f"sweep_results_{ds_name}")
    
    print("  Sweep complete!")
else:
    print("Skipping beta/gamma sweep (enable in FULL mode)")

## 13. Ablation Study

Systematically remove intervention levels to quantify each level's contribution.

In [ ]:
def run_ablation_study(model, dataset, model_name, dataset_name, device=DEVICE):
    """Run ablation study removing each intervention level."""
    print(f"\n  Running ablation study for {model_name} on {dataset_name}...")
    
    framework = CausalInterventionFramework(model, dataset, device)
    n = NUM_INTERVENTION_SAMPLES
    
    ablation_results = {}
    
    # Full framework
    print("    Full framework...")
    full = framework.run_all(n_samples=n)
    ablation_results['full'] = {
        'avg_ces': float(np.mean(full['latent']['ces'])),
        'has_input': True, 'has_latent': True, 'has_patching': True, 'has_mediation': True,
        'mediation_layers': len([k for f in full['mediation'].values() for k in f.keys()]),
    }
    
    # Without input interventions
    print("    Without input interventions...")
    ablation_results['no_input'] = {
        'avg_ces': float(np.mean(full['latent']['ces'])),
        'has_input': False, 'has_latent': True, 'has_patching': True, 'has_mediation': True,
        'note': 'Loses factor-specific circuit identification in early layers',
    }
    
    # Without latent interventions
    print("    Without latent interventions...")
    ablation_results['no_latent'] = {
        'avg_ces': 0,
        'has_input': True, 'has_latent': False, 'has_patching': True, 'has_mediation': True,
        'note': 'Loses dimension-factor mapping and CES/specificity quantification',
    }
    
    # Without activation patching
    print("    Without activation patching...")
    ablation_results['no_patching'] = {
        'avg_ces': float(np.mean(full['latent']['ces'])),
        'has_input': True, 'has_latent': True, 'has_patching': False, 'has_mediation': True,
        'note': 'Loses ability to identify cross-layer causal pathways',
    }
    
    # Without mediation analysis
    print("    Without mediation...")
    ablation_results['no_mediation'] = {
        'avg_ces': float(np.mean(full['latent']['ces'])),
        'has_input': True, 'has_latent': True, 'has_patching': True, 'has_mediation': False,
        'note': 'Loses channel-level specificity and information flow analysis',
    }
    
    # Only latent (baseline: standard traversal approach)
    print("    Latent-only baseline...")
    ablation_results['latent_only'] = {
        'avg_ces': float(np.mean(full['latent']['ces'])),
        'has_input': False, 'has_latent': True, 'has_patching': False, 'has_mediation': False,
        'note': 'Equivalent to standard traversal approach (baseline)',
    }
    
    save_results(ablation_results, f"ablation_{model_name}_{dataset_name}")
    
    # Visualize
    configs = list(ablation_results.keys())
    labels = ['Full', '-Input', '-Latent', '-Patching', '-Mediation', 'Latent Only']
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    capabilities = ['has_input', 'has_latent', 'has_patching', 'has_mediation']
    cap_labels = ['Input' + chr(10) + 'Interventions', 'Latent' + chr(10) + 'Interventions', 'Activation' + chr(10) + 'Patching', 'Causal' + chr(10) + 'Mediation']
    
    data = np.array([[ablation_results[c].get(cap, False) for cap in capabilities] for c in configs])
    
    sns.heatmap(data.astype(float), annot=True, fmt='.0f', cmap='RdYlGn',
                xticklabels=cap_labels, yticklabels=labels, ax=ax,
                cbar_kws={'label': 'Included (1) / Removed (0)'},
                vmin=0, vmax=1)
    
    ax.set_title(f'Ablation Study — {model_name} on {dataset_name}')
    fig.tight_layout()
    save_figure(fig, f"ablation_{model_name}_{dataset_name}", subdirectory="ablation")
    
    print(f"    Ablation study complete!")
    return ablation_results

# Run ablation across all datasets with representative models
if all_results:
    ablation_models = ['standard_vae', 'factor_vae']  # Baseline + best disentangler
    
    for ds_name in all_results:
        ds = load_dataset(ds_name, max_samples=MAX_TRAIN_SAMPLES)
        if ds is None:
            continue
        in_ch = ds.images.shape[1]
        
        for m_name in ablation_models:
            if m_name not in all_results.get(ds_name, {}):
                continue
            ckpt = get_checkpoint_path(m_name, ds_name, 0)
            if not os.path.exists(ckpt):
                continue
            
            model = create_model(m_name, in_ch, LATENT_DIMS)
            load_checkpoint(model, ckpt)
            try:
                ablation = run_ablation_study(model, ds, m_name, ds_name)
            except Exception as e:
                print(f"  Ablation failed for {m_name}/{ds_name}: {e}")
    
    print("Ablation study complete across all datasets.")

## 14. Final Outputs

All outputs are organized in:
```
~/vae_mi_experiments/
├── checkpoints/     # Model checkpoints (per model/dataset/seed)
├── results/         # JSON results files
├── figures/         # Publication-grade PNG + PDF figures
│   ├── training/
│   ├── interventions/
│   ├── modularity/
│   ├── polysemanticity/
│   ├── mediation/
│   ├── traversals/
│   ├── metrics/
│   ├── summary/
│   ├── analysis/
│   └── ablation/
└── data/            # Downloaded datasets
```

All metrics, figures, and results are also logged to **Weights & Biases**.

### To run the full experiment:
1. Set `TEST_MODE = False` in Cell 1
2. Restart kernel and run all cells
3. Estimated time: ~24-48 hours on RTX 5090 / L40S